In [ ]:

import sys
from pathlib import Path

print("Python version:", sys.version)
print("Notebook location:", Path.cwd())
print("Python executable:", sys.executable)


In [ ]:

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project folder exists:", PROJECT_DIR.is_dir())
print("Data folder exists:", DATA_DIR.is_dir())
print("Raw folder exists:", RAW_DIR.is_dir())
print("Processed folder exists:", PROCESSED_DIR.is_dir())


In [ ]:

import requests

url = "https://archive-api.open-meteo.com/v1/archive"

try:
    response = requests.get(
        url,
        params={
            "latitude": 30.9010,
            "longitude": 75.8573,
            "start_date": "2025-01-01",
            "end_date": "2025-01-03",
            "daily": "rain_sum",
            "timezone": "Asia/Kolkata"
        },
        timeout=(15, 60)
    )

    print("HTTP status:", response.status_code)
    response.raise_for_status()
    print("Connection successful!")
    print(response.json())

except requests.exceptions.Timeout:
    print("The server connection timed out. Please retry.")
except requests.exceptions.ConnectionError as e:
    print("Could not connect to the server.")
    print("Details:", e)
except requests.exceptions.RequestException as e:
    print("Request failed:", e)


In [ ]:

import requests

test_url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

test_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-01-01",
    "end_date": "2025-01-03",
    "hourly": "temperature_2m,precipitation",
    "timezone": "Asia/Kolkata"
}

try:
    response = requests.get(
        test_url,
        params=test_params,
        timeout=(15, 45)
    )
    print("HTTP status:", response.status_code)
    response.raise_for_status()

    test_data = response.json()
    print("SUCCESS: Historical forecast API is reachable.")
    print("Available fields:", list(test_data.get("hourly", {}).keys()))

except requests.exceptions.Timeout:
    print("TIMEOUT: This endpoint also timed out.")
except requests.exceptions.ConnectionError as error:
    print("CONNECTION ERROR:", error)
except requests.exceptions.RequestException as error:
    print("REQUEST ERROR:", error)


In [ ]:

import requests
import pandas as pd
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Historical forecast API
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "hourly": "temperature_2m,precipitation",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=(15, 120))
response.raise_for_status()

data = response.json()

# Convert hourly API data into a table
df = pd.DataFrame(data["hourly"])

# Save the raw dataset
file_path = RAW_DIR / "ludhiana_historical_forecast.csv"
df.to_csv(file_path, index=False)

print("SUCCESS! Dataset downloaded.")
print("Rows and columns:", df.shape)
print("Columns:", list(df.columns))
print("First five rows:")
display(df.head())
print("Saved to:", file_path)


In [ ]:

import requests
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "apparent_temperature,"
        "precipitation,"
        "rain,"
        "pressure_msl,"
        "cloud_cover,"
        "wind_speed_10m,"
        "wind_direction_10m,"
        "wind_gusts_10m"
    ),
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=(15, 120))
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data["hourly"])

file_path = RAW_DIR / "ludhiana_weather_features_hourly.csv"
df.to_csv(file_path, index=False)

print("SUCCESS! Expanded weather dataset downloaded.")
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values per column:")
print(df.isna().sum())
print("\nFirst five rows:")
display(df.head())
print("\nSaved to:", file_path)


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load the expanded hourly dataset
hourly_path = RAW_DIR / "ludhiana_weather_features_hourly.csv"
hourly_df = pd.read_csv(hourly_path)

# Convert timestamps to real datetime values
hourly_df["time"] = pd.to_datetime(hourly_df["time"])
hourly_df = hourly_df.sort_values("time")

# Create a daily date column
hourly_df["date"] = hourly_df["time"].dt.date

# Summarize hourly weather into daily features
daily_df = hourly_df.groupby("date").agg(
    temperature_mean=("temperature_2m", "mean"),
    temperature_max=("temperature_2m", "max"),
    temperature_min=("temperature_2m", "min"),
    humidity_mean=("relative_humidity_2m", "mean"),
    dew_point_mean=("dew_point_2m", "mean"),
    apparent_temperature_max=("apparent_temperature", "max"),
    precipitation_sum=("precipitation", "sum"),
    rain_sum=("rain", "sum"),
    pressure_mean=("pressure_msl", "mean"),
    cloud_cover_mean=("cloud_cover", "mean"),
    wind_speed_mean=("wind_speed_10m", "mean"),
    wind_speed_max=("wind_speed_10m", "max"),
    wind_gusts_max=("wind_gusts_10m", "max")
).reset_index()

# Add calendar features
daily_df["date"] = pd.to_datetime(daily_df["date"])
daily_df["month"] = daily_df["date"].dt.month
daily_df["day_of_week"] = daily_df["date"].dt.dayofweek
daily_df["day_of_year"] = daily_df["date"].dt.dayofyear

# Save the daily dataset
output_path = PROCESSED_DIR / "ludhiana_daily_weather.csv"
daily_df.to_csv(output_path, index=False)

print("SUCCESS! Hourly data converted to daily data.")
print("Daily dataset shape:", daily_df.shape)
print("Date range:", daily_df["date"].min(), "to", daily_df["date"].max())
print("Missing values:", daily_df.isna().sum().sum())
print("\nFirst five daily records:")
display(daily_df.head())
print("\nSaved to:", output_path)


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Load the daily dataset
input_path = PROCESSED_DIR / "ludhiana_daily_weather.csv"
daily_df = pd.read_csv(input_path, parse_dates=["date"])

# Ensure chronological order
daily_df = daily_df.sort_values("date").reset_index(drop=True)

# Create tomorrow's rainfall target
daily_df["tomorrow_rainfall_mm"] = daily_df["rain_sum"].shift(-1)

# Remove the last row because tomorrow's actual rainfall is not available
model_df = daily_df.dropna(subset=["tomorrow_rainfall_mm"]).copy()

# Save the dataset ready for model preparation
output_path = PROCESSED_DIR / "ludhiana_rainfall_target.csv"
model_df.to_csv(output_path, index=False)

print("SUCCESS! Tomorrow's rainfall target created.")
print("Original daily records:", len(daily_df))
print("Records ready for modelling:", len(model_df))
print("Missing target values:", model_df["tomorrow_rainfall_mm"].isna().sum())

print("\nCheck the target alignment:")
display(
    model_df[["date", "rain_sum", "tomorrow_rainfall_mm"]].head(7)
)

print("\nSaved to:", output_path)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
data_path = PROJECT_DIR / "data" / "processed" / "ludhiana_rainfall_target.csv"

df = pd.read_csv(data_path, parse_dates=["date"])
target = df["tomorrow_rainfall_mm"]

print("DATASET SUMMARY")
print("-" * 40)
print("Number of records:", len(df))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Missing values:", df.isna().sum().sum())
print("Minimum tomorrow rainfall (mm):", target.min())
print("Maximum tomorrow rainfall (mm):", target.max())
print("Average tomorrow rainfall (mm):", round(target.mean(), 2))
print("Median tomorrow rainfall (mm):", round(target.median(), 2))

print("\nRAINFALL CATEGORIES")
print("Dry days (0 mm):", (target == 0).sum())
print("Light rain (0–2.5 mm):", ((target > 0) & (target < 2.5)).sum())
print("Moderate rain (2.5–10 mm):", ((target >= 2.5) & (target < 10)).sum())
print("Heavy rain (10 mm or more):", (target >= 10).sum())

plt.figure(figsize=(10, 5))
plt.hist(target, bins=50, edgecolor="black")
plt.title("Distribution of Tomorrow's Rainfall — Ludhiana")
plt.xlabel("Tomorrow's rainfall (mm)")
plt.ylabel("Number of days")
plt.tight_layout()
plt.show()


In [ ]:

import requests
from datetime import date, timedelta

url = "https://api.open-meteo.com/v1/forecast"

tomorrow = date.today() + timedelta(days=1)

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "daily": "precipitation_sum,temperature_2m_max,temperature_2m_min",
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "precipitation,"
        "rain,"
        "pressure_msl,"
        "cloud_cover,"
        "wind_speed_10m,"
        "wind_gusts_10m"
    ),
    "forecast_days": 2,
    "timezone": "Asia/Kolkata"
}

try:
    response = requests.get(url, params=params, timeout=(15, 45))
    response.raise_for_status()
    forecast_data = response.json()

    print("LIVE FORECAST API SUCCESSFUL")
    print("Forecast dates:", forecast_data["daily"]["time"])
    print("\nDaily forecast:")
    for i, forecast_date in enumerate(forecast_data["daily"]["time"]):
        print(
            forecast_date,
            "| Rainfall:", forecast_data["daily"]["precipitation_sum"][i], "mm",
            "| Max temp:", forecast_data["daily"]["temperature_2m_max"][i], "°C",
            "| Min temp:", forecast_data["daily"]["temperature_2m_min"][i], "°C"
        )

    print("\nHourly fields received:", list(forecast_data["hourly"].keys()))

except requests.exceptions.RequestException as error:
    print("API request failed:", error)


In [ ]:

import requests
import pandas as pd

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-01-01",
    "end_date": "2025-01-07",
    "daily": (
        "precipitation_sum,"
        "rain_sum,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "relative_humidity_2m_mean,"
        "dew_point_2m_mean,"
        "cloud_cover_mean,"
        "wind_speed_10m_max"
    ),
    "timezone": "Asia/Kolkata"
}

try:
    response = requests.get(url, params=params, timeout=(15, 60))
    response.raise_for_status()
    data = response.json()

    if "daily" in data:
        daily_forecast_df = pd.DataFrame(data["daily"])
        print("SUCCESS! Historical daily forecast data received.")
        print("Shape:", daily_forecast_df.shape)
        print("Columns:", daily_forecast_df.columns.tolist())
        display(daily_forecast_df.head())
    else:
        print("The API responded, but no daily data was returned.")
        print(data)

except requests.exceptions.RequestException as error:
    print("Request failed:", error)


In [ ]:

import requests
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "daily": (
        "precipitation_sum,"
        "rain_sum,"
        "temperature_2m_max,"
        "temperature_2m_min,"
        "relative_humidity_2m_mean,"
        "dew_point_2m_mean,"
        "cloud_cover_mean,"
        "wind_speed_10m_max"
    ),
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=(15, 120))
response.raise_for_status()

data = response.json()
forecast_df = pd.DataFrame(data["daily"])

# Standardize the date column
forecast_df = forecast_df.rename(columns={"time": "date"})
forecast_df["date"] = pd.to_datetime(forecast_df["date"])
forecast_df = forecast_df.sort_values("date").reset_index(drop=True)

# Inspect missing values
print("Downloaded historical daily forecast data.")
print("Shape:", forecast_df.shape)
print("Date range:", forecast_df["date"].min(), "to", forecast_df["date"].max())

print("\nMissing values per column:")
print(forecast_df.isna().sum())

# Save without overwriting the earlier datasets
output_path = RAW_DIR / "ludhiana_historical_daily_forecast.csv"
forecast_df.to_csv(output_path, index=False)

print("\nFirst five rows:")
display(forecast_df.head())

print("\nSaved to:", output_path)


In [ ]:

import requests
import pandas as pd

url = "https://previous-runs-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-03-14",
    "end_date": "2025-03-20",
    "hourly": [
        "temperature_2m_previous_day1",
        "relative_humidity_2m_previous_day1",
        "dew_point_2m_previous_day1",
        "precipitation_previous_day1",
        "rain_previous_day1",
        "cloud_cover_previous_day1",
        "wind_speed_10m_previous_day1"
    ],
    "timezone": "Asia/Kolkata",
    "models": "ecmwf_ifs025"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)

response.raise_for_status()
data = response.json()

print("\nResponse timezone:", data.get("timezone"))
print("\nAvailable response sections:", list(data.keys()))

if "hourly" in data:
    test_df = pd.DataFrame(data["hourly"])
    print("\nDataset shape:", test_df.shape)
    display(test_df.head(10))
    print("\nMissing values:")
    display(test_df.isna().sum())
else:
    print("\nAPI response:")
    print(data)


In [ ]:

import requests
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = "https://previous-runs-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "hourly": [
        "temperature_2m_previous_day1",
        "relative_humidity_2m_previous_day1",
        "dew_point_2m_previous_day1",
        "precipitation_previous_day1",
        "cloud_cover_previous_day1",
        "wind_speed_10m_previous_day1"
    ],
    "timezone": "Asia/Kolkata",
    "models": "ecmwf_ifs025"
}

response = requests.get(url, params=params, timeout=120)
print("HTTP status:", response.status_code)
response.raise_for_status()

data = response.json()
hourly_df = pd.DataFrame(data["hourly"])

output_path = RAW_DIR / "ludhiana_previous_day1_forecasts_hourly.csv"
hourly_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Dataset shape:", hourly_df.shape)
print("\nDate range:", hourly_df["time"].min(), "to", hourly_df["time"].max())
print("\nMissing values:")
display(hourly_df.isna().sum())


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

file_path = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "ludhiana_previous_day1_forecasts_hourly.csv"
)

hourly_df = pd.read_csv(file_path, parse_dates=["time"])

feature_cols = [
    col for col in hourly_df.columns
    if col != "time"
]

hourly_df["year"] = hourly_df["time"].dt.year

print("HOURLY COVERAGE BY YEAR")
yearly_coverage = hourly_df.groupby("year")[feature_cols].agg(
    lambda col: col.notna().mean() * 100
)
display(yearly_coverage.round(1))

print("\nROWS WITH ALL SIX FEATURES AVAILABLE")
complete_rows = hourly_df[feature_cols].notna().all(axis=1)
print(f"{complete_rows.sum():,} / {len(hourly_df):,}")
print(f"Coverage: {complete_rows.mean() * 100:.1f}%")

print("\nFIRST AND LAST AVAILABLE FORECAST VALUES")
for col in feature_cols:
    available = hourly_df.loc[hourly_df[col].notna(), "time"]
    if len(available):
        print(
            col,
            "| first:", available.min(),
            "| last:", available.max()
        )
    else:
        print(col, "| no available values")


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

file_path = RAW_DIR / "ludhiana_previous_day1_forecasts_hourly.csv"
hourly_df = pd.read_csv(file_path, parse_dates=["time"])

feature_map = {
    "temperature_2m_previous_day1": "temperature_mean_c",
    "relative_humidity_2m_previous_day1": "humidity_mean_pct",
    "dew_point_2m_previous_day1": "dew_point_mean_c",
    "precipitation_previous_day1": "precipitation_sum_mm",
    "cloud_cover_previous_day1": "cloud_cover_mean_pct",
    "wind_speed_10m_previous_day1": "wind_speed_mean_kmh"
}

hourly_df = hourly_df.rename(columns=feature_map)

# Keep only 2025, when all six forecast features are available.
hourly_df = hourly_df[
    hourly_df["time"].dt.year == 2025
].copy()

feature_cols = list(feature_map.values())

print("Missing values before aggregation:")
display(hourly_df[feature_cols].isna().sum())

# Require complete hourly coverage for each day.
hourly_df["date"] = hourly_df["time"].dt.normalize()

daily = hourly_df.groupby("date").agg(
    hours_available=("time", "count"),
    temperature_mean_c=("temperature_mean_c", "mean"),
    humidity_mean_pct=("humidity_mean_pct", "mean"),
    dew_point_mean_c=("dew_point_mean_c", "mean"),
    precipitation_sum_mm=("precipitation_sum_mm", "sum"),
    cloud_cover_mean_pct=("cloud_cover_mean_pct", "mean"),
    wind_speed_mean_kmh=("wind_speed_mean_kmh", "mean")
).reset_index()

daily = daily[daily["hours_available"] == 24].copy()

output_path = PROCESSED_DIR / "ludhiana_2025_day_ahead_forecasts.csv"
daily.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("Daily dataset shape:", daily.shape)
print("Date range:", daily["date"].min(), "to", daily["date"].max())
print("\nFirst 5 days:")
display(daily.head())
print("\nMissing values:")
display(daily.isna().sum())


In [ ]:

import requests
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "daily": ["precipitation_sum", "rain_sum"],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=120)
print("HTTP status:", response.status_code)
response.raise_for_status()

data = response.json()

observed_df = pd.DataFrame({
    "date": pd.to_datetime(data["daily"]["time"]),
    "observed_precipitation_mm": data["daily"]["precipitation_sum"],
    "observed_rainfall_mm": data["daily"]["rain_sum"]
})

output_path = RAW_DIR / "ludhiana_2025_observed_rainfall.csv"
observed_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", observed_df.shape)
print("Date range:", observed_df["date"].min(), "to", observed_df["date"].max())
print("\nMissing values:")
display(observed_df.isna().sum())
print("\nFirst 5 records:")
display(observed_df.head())
print("\nRainfall summary:")
display(observed_df["observed_rainfall_mm"].describe().round(2))


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"

forecast_path = PROCESSED_DIR / "ludhiana_2025_day_ahead_forecasts.csv"
rainfall_path = RAW_DIR / "ludhiana_2025_observed_rainfall.csv"

forecasts = pd.read_csv(forecast_path, parse_dates=["date"])
rainfall = pd.read_csv(rainfall_path, parse_dates=["date"])

# Match forecast valid dates to the corresponding rainfall dates.
comparison = forecasts.merge(
    rainfall,
    on="date",
    how="inner",
    validate="one_to_one"
)

print("Matched days:", len(comparison))
print("Date range:", comparison["date"].min().date(), "to",
      comparison["date"].max().date())

print("\nMissing values:")
display(comparison.isna().sum())

# Compare forecast precipitation with the reference rainfall series.
actual = comparison["observed_rainfall_mm"]
predicted = comparison["precipitation_sum_mm"]

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))

print("\nDAILY PRECIPITATION COMPARISON")
print(f"MAE:  {mae:.3f} mm")
print(f"RMSE: {rmse:.3f} mm")

print("\nFirst 10 matched days:")
display(
    comparison[
        [
            "date",
            "precipitation_sum_mm",
            "observed_rainfall_mm",
            "temperature_mean_c",
            "humidity_mean_pct"
        ]
    ].head(10).round(2)
)

output_path = PROCESSED_DIR / "ludhiana_2025_forecast_rainfall_comparison.csv"
comparison.to_csv(output_path, index=False)
print("\nSaved:", output_path)


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

comparison_path = (
    PROCESSED_DIR / "ludhiana_2025_forecast_rainfall_comparison.csv"
)

df = pd.read_csv(comparison_path, parse_dates=["date"])

feature_columns = [
    "temperature_mean_c",
    "humidity_mean_pct",
    "dew_point_mean_c",
    "cloud_cover_mean_pct",
    "wind_speed_mean_kmh"
]

target_column = "observed_rainfall_mm"

# Keep only model inputs and the rainfall reference.
training_df = df[["date"] + feature_columns + [target_column]].copy()

# Check data quality.
assert training_df["date"].is_monotonic_increasing
assert training_df["date"].is_unique
assert training_df[feature_columns + [target_column]].notna().all().all()

output_path = PROCESSED_DIR / "ludhiana_2025_ml_training_data.csv"
training_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", training_df.shape)
print("Features:", feature_columns)
print("Target:", target_column)
print("\nDate range:")
print(training_df["date"].min().date(), "to",
      training_df["date"].max().date())

print("\nTarget rainfall summary:")
display(training_df[target_column].describe().round(2))

print("\nFirst 5 rows:")
display(training_df.head())


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
file_path = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "ludhiana_2025_ml_training_data.csv"
)

check_df = pd.read_csv(file_path, parse_dates=["date"])

print("Column names:")
print(check_df.columns.tolist())

print("\nFirst row — values listed by column:")
for column in check_df.columns:
    print(f"{column}: {check_df.loc[0, column]}")

print("\nLast row — values listed by column:")
for column in check_df.columns:
    print(f"{column}: {check_df.loc[len(check_df) - 1, column]}")

print("\nFirst 5 rows, with dates as the index:")
display(check_df.head().set_index("date").T)

print("\nMissing values:")
display(check_df.isna().sum())


In [ ]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-11-01",
    "end_date": "2025-12-31",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=30)

print("HTTP status:", response.status_code)
response.raise_for_status()

data = response.json()
archive_check = pd.DataFrame(data["daily"])
archive_check["time"] = pd.to_datetime(archive_check["time"])

print("\nDaily data returned:", len(archive_check))
print("\nMonthly totals:")
archive_check["month"] = archive_check["time"].dt.to_period("M")

display(
    archive_check.groupby("month")[
        ["precipitation_sum", "rain_sum"]
    ].agg(["count", "sum", "max"]).round(2)
)

print("\nDays with rainfall greater than zero:")
display(
    archive_check.loc[
        (archive_check["precipitation_sum"] > 0) |
        (archive_check["rain_sum"] > 0)
    ].head(20)
)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

forecast_path = (
    PROJECT_DIR / "data" / "processed" /
    "ludhiana_2025_day_ahead_forecasts.csv"
)

rain_path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_2025_observed_rainfall.csv"
)

forecast = pd.read_csv(forecast_path, parse_dates=["date"])
rain = pd.read_csv(rain_path, parse_dates=["date"])

print("Forecast columns:")
print(forecast.columns.tolist())

print("\nRainfall columns:")
print(rain.columns.tolist())

comparison = forecast.merge(
    rain,
    on="date",
    how="inner",
    validate="one_to_one"
)

december = comparison.loc[
    comparison["date"].between("2025-12-10", "2025-12-31")
].copy()

december["date"] = december["date"].dt.strftime("%Y-%m-%d")

print("\nDecember source-data comparison:")
print(december.to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

PROJECT_DIR = Path.cwd().parent

forecast = pd.read_csv(
    PROJECT_DIR / "data" / "processed" /
    "ludhiana_2025_day_ahead_forecasts.csv",
    parse_dates=["date"]
)

rain = pd.read_csv(
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_2025_observed_rainfall.csv",
    parse_dates=["date"]
)

comparison = forecast.merge(
    rain,
    on="date",
    how="inner",
    validate="one_to_one"
)

print("Matched dates:", len(comparison))
print("Forecast dates:", comparison["date"].nunique())
print("Rainfall dates:", rain["date"].nunique())

print("\nNon-zero daily values:")
print(
    "Forecast precipitation:",
    (comparison["precipitation_sum_mm"] > 0).sum()
)
print(
    "Archive precipitation:",
    (comparison["observed_precipitation_mm"] > 0).sum()
)
print(
    "Archive rainfall:",
    (comparison["observed_rainfall_mm"] > 0).sum()
)

print("\nMonthly totals (mm):")
comparison["month"] = comparison["date"].dt.to_period("M")

monthly = comparison.groupby("month").agg(
    forecast_precipitation_mm=("precipitation_sum_mm", "sum"),
    archive_precipitation_mm=("observed_precipitation_mm", "sum"),
    archive_rainfall_mm=("observed_rainfall_mm", "sum")
)

print(monthly.round(2).to_string())

print("\nFull-year direct precipitation comparison:")
mae = mean_absolute_error(
    comparison["observed_precipitation_mm"],
    comparison["precipitation_sum_mm"]
)
rmse = np.sqrt(mean_squared_error(
    comparison["observed_precipitation_mm"],
    comparison["precipitation_sum_mm"]
))

print("MAE:", round(mae, 3), "mm")
print("RMSE:", round(rmse, 3), "mm")

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

df = pd.read_csv(path, parse_dates=["date"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head().to_string(index=False))

print("\nLast 5 rows:")
print(df.tail().to_string(index=False))

print("\nMissing values:")
print(df.isna().sum().to_string())

print("\nRows per year:")
print(df.groupby(df["date"].dt.year).size().to_string())

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

print("File exists:", path.exists())
print("File path:", path)

if path.exists():
    print("\nFirst 10 lines of the original file:")
    with open(path, "r", encoding="utf-8-sig") as file:
        for _ in range(10):
            line = file.readline()
            if not line:
                break
            print(repr(line))

    print("\nColumns detected by pandas:")
    preview = pd.read_csv(path, nrows=5)
    print(preview.columns.tolist())
    display(preview)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

df = pd.read_csv(path)

# The original timestamp column is called "time"
df["time"] = pd.to_datetime(df["time"])
df = df.rename(columns={"time": "date"})

weather_columns = [
    col for col in df.columns if col != "date"
]

print("Dataset shape:", df.shape)
print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

print("\nPercentage of available values by year:")
df["year"] = df["date"].dt.year

coverage = (
    df.groupby("year")[weather_columns]
    .apply(lambda group: group.notna().mean() * 100)
)

display(coverage.round(1))

print("\nPercentage of rows with all weather features available:")
complete_rows = df[weather_columns].notna().all(axis=1)

print(round(complete_rows.mean() * 100, 2), "%")
print("Complete rows:", int(complete_rows.sum()))
print("Total rows:", len(df))

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

df = pd.read_csv(path)
df["time"] = pd.to_datetime(df["time"])

weather_columns = [
    col for col in df.columns if col != "time"
]

# Select 2025 only
df_2025 = df.loc[df["time"].dt.year == 2025].copy()

print("2025 rows:", len(df_2025))
print("Expected hourly rows:", 365 * 24)

print("\nMissing values in 2025:")
print(df_2025[weather_columns].isna().sum().to_string())

print("\nFirst 5 available hourly records:")
display(df_2025.head().round(2))

print("\nLast 5 hourly records:")
display(df_2025.tail().round(2))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

hourly_path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

daily_path = (
    PROJECT_DIR / "data" / "processed" /
    "ludhiana_2025_day_ahead_forecasts.csv"
)

hourly = pd.read_csv(hourly_path, parse_dates=["time"])
daily = pd.read_csv(daily_path, parse_dates=["date"])

# Keep 2025 hourly records
hourly_2025 = hourly.loc[
    hourly["time"].dt.year == 2025
].copy()

# Rebuild daily features from hourly data
rebuilt = (
    hourly_2025.assign(date=hourly_2025["time"].dt.normalize())
    .groupby("date")
    .agg(
        hours_available=("time", "count"),
        temperature_mean_c=("temperature_2m_previous_day1", "mean"),
        humidity_mean_pct=("relative_humidity_2m_previous_day1", "mean"),
        dew_point_mean_c=("dew_point_2m_previous_day1", "mean"),
        precipitation_sum_mm=("precipitation_previous_day1", "sum"),
        cloud_cover_mean_pct=("cloud_cover_previous_day1", "mean"),
        wind_speed_mean_kmh=("wind_speed_10m_previous_day1", "mean")
    )
    .reset_index()
)

print("Rebuilt daily rows:", len(rebuilt))
print("Existing daily rows:", len(daily))
print("Days with 24 hourly records:", int((rebuilt["hours_available"] == 24).sum()))

comparison = rebuilt.merge(
    daily,
    on="date",
    how="outer",
    suffixes=("_rebuilt", "_saved"),
    indicator=True
)

print("\nDate matching:")
print(comparison["_merge"].value_counts().to_string())

columns_to_compare = [
    "temperature_mean_c",
    "humidity_mean_pct",
    "dew_point_mean_c",
    "precipitation_sum_mm",
    "cloud_cover_mean_pct",
    "wind_speed_mean_kmh"
]

print("\nMaximum absolute difference for each feature:")
for col in columns_to_compare:
    difference = (
        comparison[f"{col}_rebuilt"] -
        comparison[f"{col}_saved"]
    ).abs()
    print(f"{col}: {difference.max():.6f}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_DIR = Path.cwd().parent

# Load the same-date comparison dataset
data_path = (
    PROJECT_DIR / "data" / "processed" /
    "ludhiana_2025_forecast_rainfall_comparison.csv"
)

comparison = pd.read_csv(data_path, parse_dates=["date"])

print("Dataset shape:", comparison.shape)
print("\nColumn names:")
print(comparison.columns.tolist())

print("\nFirst five rows:")
display(comparison.head())

print("\nDate range:")
print(comparison["date"].min(), "to", comparison["date"].max())

In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
raw_dir = PROJECT_DIR / "data" / "raw"
processed_dir = PROJECT_DIR / "data" / "processed"

files_to_check = [
    raw_dir / "ludhiana_historical_forecast.csv",
    raw_dir / "ludhiana_weather_features_hourly.csv",
    raw_dir / "ludhiana_previous_day1_forecasts_hourly.csv",
    raw_dir / "ludhiana_2025_observed_rainfall.csv",
]

for file_path in files_to_check:
    print("\n" + "=" * 60)
    print("File:", file_path.name)

    if not file_path.exists():
        print("STATUS: File not found")
        continue

    df = pd.read_csv(file_path)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    date_col = next(
        (c for c in ["date", "time", "datetime"]
         if c in df.columns),
        None
    )

    if date_col:
        dates = pd.to_datetime(df[date_col], errors="coerce")
        print("Date range:", dates.min(), "to", dates.max())
        print("Year counts:")
        print(dates.dt.year.value_counts().sort_index().to_dict())


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

forecast_path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

forecast = pd.read_csv(forecast_path, parse_dates=["time"])

forecast_columns = [
    "temperature_2m_previous_day1",
    "relative_humidity_2m_previous_day1",
    "dew_point_2m_previous_day1",
    "precipitation_previous_day1",
    "cloud_cover_previous_day1",
    "wind_speed_10m_previous_day1",
]

forecast["year"] = forecast["time"].dt.year

# Count complete rows separately for each year
yearly_report = (
    forecast.groupby("year")
    .apply(
        lambda group: pd.Series({
            "total_rows": len(group),
            "complete_rows": group[forecast_columns]
                .notna().all(axis=1).sum()
        }),
        include_groups=False
    )
)

print("Forecast feature availability by year:")
display(yearly_report.astype(int))

print("\nMissing percentage for each forecast feature:")
display(
    (forecast[forecast_columns].isna().mean() * 100).round(2)
)


In [ ]:

# Check complete forecast-feature coverage by month

forecast["year_month"] = forecast["time"].dt.to_period("M")

monthly_coverage = (
    forecast.groupby("year_month")
    .apply(
        lambda group: pd.Series({
            "total_hours": len(group),
            "complete_hours": group[forecast_columns]
                .notna().all(axis=1).sum(),
            "coverage_percent": round(
                group[forecast_columns].notna().all(axis=1).mean() * 100,
                2
            )
        }),
        include_groups=False
    )
)

print("Monthly forecast-feature coverage:")
display(monthly_coverage.head(24))

print("\nLast 24 months:")
display(monthly_coverage.tail(24))


In [ ]:

import requests
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)
response.raise_for_status()

weather = response.json()

rainfall = pd.DataFrame({
    "date": pd.to_datetime(weather["daily"]["time"]),
    "observed_precipitation_mm": weather["daily"]["precipitation_sum"],
    "observed_rainfall_mm": weather["daily"]["rain_sum"]
})

output_path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_rainfall_reference_2022_2025.csv"
)

rainfall.to_csv(output_path, index=False)

rainfall["year"] = rainfall["date"].dt.year

yearly_summary = rainfall.groupby("year").agg(
    days=("date", "count"),
    missing_rainfall=("observed_rainfall_mm", lambda x: x.isna().sum()),
    rainy_days=("observed_rainfall_mm", lambda x: (x > 0).sum()),
    total_rainfall_mm=("observed_rainfall_mm", "sum")
).round(2)

print("\nDate range:", rainfall["date"].min(), "to", rainfall["date"].max())
print("Total rows:", len(rainfall))
print("Saved to:", output_path)

display(yearly_summary)


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

# Load hourly day-ahead forecast features
forecast_path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_previous_day1_forecasts_hourly.csv"
)

forecast = pd.read_csv(forecast_path, parse_dates=["time"])

# Convert hourly forecast features into daily features
forecast["date"] = forecast["time"].dt.normalize()

feature_columns = [
    "temperature_2m_previous_day1",
    "relative_humidity_2m_previous_day1",
    "dew_point_2m_previous_day1",
    "precipitation_previous_day1",
    "cloud_cover_previous_day1",
    "wind_speed_10m_previous_day1",
]

daily_forecast = (
    forecast.groupby("date")[feature_columns]
    .mean()
    .reset_index()
)

# Load four-year historical rainfall reference
rainfall_path = (
    PROJECT_DIR / "data" / "raw" /
    "ludhiana_rainfall_reference_2022_2025.csv"
)

rainfall = pd.read_csv(rainfall_path, parse_dates=["date"])

# Match forecasts and rainfall by the same calendar date
dataset = daily_forecast.merge(
    rainfall[["date", "observed_rainfall_mm"]],
    on="date",
    how="inner",
    validate="one_to_one"
)

# Keep only rows with all required inputs and target
dataset = dataset.dropna().sort_values("date").reset_index(drop=True)

# Save the clean dataset
output_path = (
    PROJECT_DIR / "data" / "processed" /
    "ludhiana_day_ahead_training_data_2022_2025.csv"
)

dataset.to_csv(output_path, index=False)

print("Dataset shape:", dataset.shape)
print("Date range:", dataset["date"].min(), "to", dataset["date"].max())
print("Saved to:", output_path)

print("\nUsable rows by year:")
display(dataset.groupby(dataset["date"].dt.year).size().rename("usable_days"))

print("\nMissing values:")
display(dataset.isna().sum())

print("\nFirst five rows:")
display(dataset.head())


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

file_path = (
    PROJECT_DIR / "data" / "processed" /
    "ludhiana_day_ahead_training_data_2022_2025.csv"
)

df = pd.read_csv(file_path, parse_dates=["date"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum())

print("\nFirst 5 rows:")
display(df.head().to_string(index=False))

print("\nLast 5 rows:")
display(df.tail().to_string(index=False))

print("\nUsable rows by year:")
display(df.groupby(df["date"].dt.year).size())

print("\nDuplicate dates:", df["date"].duplicated().sum())


In [ ]:

import requests
from datetime import datetime, timezone

url = "https://previous-runs-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "hourly": (
        "temperature_2m_previous_day1,"
        "precipitation_previous_day1"
    ),
    "models": "ecmwf_ifs025",
    "start_date": "2025-07-15",
    "end_date": "2025-07-17",
    "timezone": "UTC"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)
response.raise_for_status()

result = response.json()

print("\nAPI response keys:")
print(result.keys())

print("\nModel:")
print(result.get("model"))

print("\nTimezone:")
print(result.get("timezone"))

hourly = result.get("hourly", {})
print("\nFirst 10 timestamps and forecast values:")

for i in range(min(10, len(hourly.get("time", [])))):
    print(
        hourly["time"][i],
        "Temperature:", hourly["temperature_2m_previous_day1"][i],
        "Precipitation:", hourly["precipitation_previous_day1"][i]
    )


In [ ]:

import pandas as pd

hourly_df = pd.DataFrame({
    "time": pd.to_datetime(result["hourly"]["time"]),
    "temperature_c": result["hourly"]["temperature_2m_previous_day1"],
    "precipitation_mm": result["hourly"]["precipitation_previous_day1"],
})

print("Number of hourly records:", len(hourly_df))
print("First timestamp:", hourly_df["time"].min())
print("Last timestamp:", hourly_df["time"].max())

print("\nHourly precipitation summary:")
display(hourly_df["precipitation_mm"].describe())

print("\nDaily precipitation totals:")
display(
    hourly_df.assign(date=hourly_df["time"].dt.date)
    .groupby("date")
    .agg(
        hours=("precipitation_mm", "size"),
        precipitation_total_mm=("precipitation_mm", "sum"),
        temperature_mean_c=("temperature_c", "mean"),
    )
    .round(2)
)

print("\nAPI URL:")
print(response.url)


In [ ]:
# Step: Check hourly forecast data quality

print("Missing values:")
display(hourly_df.isnull().sum())

print("\nDuplicate timestamps:")
print(hourly_df["time"].duplicated().sum())

print("\nHourly precipitation values:")
display(
    hourly_df[
        ["time", "temperature_c", "precipitation_mm"]
    ].head(10)
)

print("\nCheck hours per day:")
daily_check = hourly_df.groupby(
    hourly_df["time"].dt.date
).size()

display(daily_check)

In [ ]:
# Step: Display the exact dates and forecast values

display(
    hourly_df.groupby(
        hourly_df["time"].dt.date
    ).agg(
        first_temperature=("temperature_c", "first"),
        last_temperature=("temperature_c", "last"),
        first_precipitation=("precipitation_mm", "first"),
        maximum_precipitation=("precipitation_mm", "max"),
        total_precipitation=("precipitation_mm", "sum"),
    ).round(2)
)

print("\nImportant:")
print("These are previous_day1 forecast values.")
print("They are not confirmed observed rainfall measurements.")
print("We must verify forecast issue-time alignment before model training.")

In [ ]:

import requests
import pandas as pd

url = "https://single-runs-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "run": "2025-07-14T00:00",
    "models": "ecmwf_ifs025",
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "precipitation,"
        "precipitation_probability,"
        "cloud_cover,"
        "wind_speed_10m"
    ),
    "timezone": "Asia/Kolkata",
    "forecast_days": 7
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    result_single = response.json()

    print("Response keys:", list(result_single.keys()))
    print("Timezone:", result_single.get("timezone"))

    hourly = result_single.get("hourly", {})
    print("Number of hourly records:", len(hourly.get("time", [])))

    if hourly.get("time"):
        forecast_df = pd.DataFrame(hourly)
        forecast_df["time"] = pd.to_datetime(forecast_df["time"])

        print("\nFirst forecast timestamps:")
        display(forecast_df.head())

        print("\nLast forecast timestamp:")
        print(forecast_df["time"].max())

        print("\nForecast URL:")
        print(response.url)
else:
    print(response.text[:2000])


In [ ]:

import requests

test_url = "https://single-runs-api.open-meteo.com/v1/forecast"

test_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "run": "2025-07-14T00:00",
    "models": "ecmwf_ifs",
    "hourly": "temperature_2m,precipitation",
    "forecast_days": 3,
    "timezone": "GMT"
}

test_response = requests.get(
    test_url,
    params=test_params,
    timeout=60
)

print("HTTP status:", test_response.status_code)
print("Response:", test_response.text[:1500])


In [ ]:

import pandas as pd

# Read the complete API response
run_data = test_response.json()

# Convert hourly forecast into a DataFrame
run_df = pd.DataFrame(run_data["hourly"])
run_df["time"] = pd.to_datetime(run_df["time"])
run_df["date"] = run_df["time"].dt.date

print("Total hourly records:", len(run_df))
print("First timestamp:", run_df["time"].min())
print("Last timestamp:", run_df["time"].max())

print("\nDaily forecast summary:")

daily_run_df = (
    run_df.groupby("date")
    .agg(
        hours=("time", "count"),
        temperature_mean_c=("temperature_2m", "mean"),
        forecast_precipitation_mm=("precipitation", "sum"),
        maximum_hourly_precipitation_mm=("precipitation", "max"),
    )
    .round(2)
)

display(daily_run_df)

print("\nForecast run requested: 2025-07-14 00:00 UTC")
print("Model: ecmwf_ifs")


In [ ]:

import requests
import pandas as pd

archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-14",
    "end_date": "2025-07-16",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "UTC"
}

archive_response = requests.get(
    archive_url,
    params=archive_params,
    timeout=60
)

print("HTTP status:", archive_response.status_code)

if archive_response.status_code == 200:
    archive_data = archive_response.json()

    observed_df = pd.DataFrame({
        "date": pd.to_datetime(archive_data["daily"]["time"]).date,
        "reference_precipitation_mm": archive_data["daily"]["precipitation_sum"],
        "reference_rainfall_mm": archive_data["daily"]["rain_sum"]
    })

    print("\nHistorical rainfall reference:")
    display(observed_df)

    print(
        "\nNote: Open-Meteo archive values are gridded historical "
        "weather estimates, not guaranteed ground-station observations."
    )
else:
    print(archive_response.text[:1500])


In [ ]:

# Merge forecasts and historical rainfall by date

forecast_daily = daily_run_df.reset_index().copy()

forecast_daily["date"] = pd.to_datetime(
    forecast_daily["date"]
).dt.date

observed_df["date"] = pd.to_datetime(
    observed_df["date"]
).dt.date

comparison_df = forecast_daily.merge(
    observed_df,
    on="date",
    how="inner",
    validate="one_to_one"
)

comparison_df["absolute_error_mm"] = (
    comparison_df["forecast_precipitation_mm"]
    - comparison_df["reference_precipitation_mm"]
).abs()

print("Comparison dataset shape:", comparison_df.shape)
display(comparison_df)

print("\nMean absolute difference (mm):")
print(round(comparison_df["absolute_error_mm"].mean(), 3))


In [ ]:

import requests
import pandas as pd
import numpy as np

# Request one historical forecast run
run_url = "https://single-runs-api.open-meteo.com/v1/forecast"

run_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "run": "2025-07-01T00:00",
    "models": "ecmwf_ifs",
    "hourly": "precipitation",
    "forecast_days": 7,
    "timezone": "GMT"
}

run_response = requests.get(
    run_url,
    params=run_params,
    timeout=60
)

print("Forecast API status:", run_response.status_code)

if run_response.status_code == 200:
    run_json = run_response.json()

    month_forecast = pd.DataFrame({
        "time": pd.to_datetime(run_json["hourly"]["time"]),
        "forecast_precipitation_mm": run_json["hourly"]["precipitation"]
    })

    month_forecast["date"] = month_forecast["time"].dt.date

    daily_forecast = (
        month_forecast.groupby("date")
        .agg(
            hours=("time", "count"),
            forecast_precipitation_mm=(
                "forecast_precipitation_mm", "sum"
            )
        )
        .reset_index()
    )

    print("Forecast dates:", daily_forecast["date"].min(),
          "to", daily_forecast["date"].max())
    display(daily_forecast)

else:
    print(run_response.text[:1500])


In [ ]:

archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-07-07",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "GMT"
}

archive_response = requests.get(
    archive_url,
    params=archive_params,
    timeout=60
)

print("Archive API status:", archive_response.status_code)

if archive_response.status_code == 200:
    archive_json = archive_response.json()

    week_observed = pd.DataFrame({
        "date": pd.to_datetime(archive_json["daily"]["time"]).date,
        "reference_precipitation_mm": (
            archive_json["daily"]["precipitation_sum"]
        ),
        "reference_rainfall_mm": archive_json["daily"]["rain_sum"]
    })

    week_comparison = daily_forecast.merge(
        week_observed,
        on="date",
        how="inner",
        validate="one_to_one"
    )

    week_comparison["absolute_error_mm"] = (
        week_comparison["forecast_precipitation_mm"]
        - week_comparison["reference_precipitation_mm"]
    ).abs()

    print("\nSeven-day comparison:")
    display(week_comparison.round(2))

    print("\nMean absolute difference (mm):")
    print(round(week_comparison["absolute_error_mm"].mean(), 3))

else:
    print(archive_response.text[:1500])


In [ ]:

# Test several historical forecast initialization dates

test_runs = [
    "2025-07-01T00:00",
    "2025-07-08T00:00",
    "2025-07-15T00:00",
    "2025-07-22T00:00",
]

run_check_results = []

for run_time in test_runs:
    params = {
        "latitude": 30.9010,
        "longitude": 75.8573,
        "run": run_time,
        "models": "ecmwf_ifs",
        "hourly": "precipitation",
        "forecast_days": 7,
        "timezone": "GMT",
    }

    try:
        response_check = requests.get(
            "https://single-runs-api.open-meteo.com/v1/forecast",
            params=params,
            timeout=60,
        )

        result_check = {
            "requested_run": run_time,
            "http_status": response_check.status_code,
            "first_forecast_time": None,
            "last_forecast_time": None,
            "hourly_records": 0,
            "error": None,
        }

        if response_check.status_code == 200:
            hourly_times = response_check.json()["hourly"]["time"]
            result_check["hourly_records"] = len(hourly_times)

            if hourly_times:
                result_check["first_forecast_time"] = hourly_times[0]
                result_check["last_forecast_time"] = hourly_times[-1]
        else:
            result_check["error"] = response_check.text[:200]

        run_check_results.append(result_check)

    except requests.RequestException as exc:
        run_check_results.append({
            "requested_run": run_time,
            "http_status": None,
            "first_forecast_time": None,
            "last_forecast_time": None,
            "hourly_records": 0,
            "error": str(exc),
        })

run_check_df = pd.DataFrame(run_check_results)
display(run_check_df)


In [ ]:

from pathlib import Path
import requests
import pandas as pd

# Project root: folder containing data/raw
project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

run_dates = [
    "2025-07-01T00:00",
    "2025-07-08T00:00",
    "2025-07-15T00:00",
    "2025-07-22T00:00",
]

all_runs = []

for run_time in run_dates:
    params = {
        "latitude": 30.9010,
        "longitude": 75.8573,
        "run": run_time,
        "models": "ecmwf_ifs",
        "hourly": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "dew_point_2m,"
            "precipitation,"
            "cloud_cover,"
            "wind_speed_10m"
        ),
        "forecast_days": 7,
        "timezone": "GMT",
    }

    r = requests.get(
        "https://single-runs-api.open-meteo.com/v1/forecast",
        params=params,
        timeout=60,
    )
    r.raise_for_status()

    hourly = pd.DataFrame(r.json()["hourly"])
    hourly["valid_time_utc"] = pd.to_datetime(hourly.pop("time"))
    hourly["run_time_utc"] = pd.to_datetime(run_time)

    all_runs.append(hourly)

forecast_runs_df = pd.concat(all_runs, ignore_index=True)

output_file = raw_dir / "ludhiana_july_2025_historical_forecast_runs.csv"
forecast_runs_df.to_csv(output_file, index=False)

print("Saved to:", output_file)
print("Rows:", len(forecast_runs_df))
print("Distinct forecast runs:", forecast_runs_df["run_time_utc"].nunique())
print("Missing values:")
display(forecast_runs_df.isna().sum())

print("\nFirst records:")
display(forecast_runs_df.head())

print("\nRecords per forecast run:")
display(
    forecast_runs_df.groupby("run_time_utc").size().rename("hourly_records")
)


In [ ]:

# Reload the saved CSV and verify its actual columns

saved_forecasts = pd.read_csv(
    output_file,
    parse_dates=["valid_time_utc", "run_time_utc"]
)

print("Dataset shape:", saved_forecasts.shape)

print("\nActual column names:")
print(saved_forecasts.columns.tolist())

print("\nFirst 5 rows:")
display(saved_forecasts.head())

print("\nRows with missing precipitation:")
display(
    saved_forecasts.loc[
        saved_forecasts["precipitation"].isna()
    ]
)

print("\nDuplicate run-time and valid-time pairs:")
print(
    saved_forecasts.duplicated(
        subset=["run_time_utc", "valid_time_utc"]
    ).sum()
)

print("\nValid date coverage per run:")
display(
    saved_forecasts.groupby("run_time_utc").agg(
        first_valid_time=("valid_time_utc", "min"),
        last_valid_time=("valid_time_utc", "max"),
        row_count=("valid_time_utc", "size"),
    )
)


In [ ]:

# Create a cleaned copy without changing the original CSV

clean_forecasts = saved_forecasts.copy()

print("Rows before cleaning:", len(clean_forecasts))
print("Missing precipitation before cleaning:",
      clean_forecasts["precipitation"].isna().sum())

# Remove only rows whose precipitation target is missing
clean_forecasts = clean_forecasts.dropna(
    subset=["precipitation"]
).copy()

clean_file = raw_dir / "ludhiana_july_2025_historical_forecast_runs_clean.csv"
clean_forecasts.to_csv(clean_file, index=False)

print("\nRows after cleaning:", len(clean_forecasts))
print("Missing values after cleaning:")
display(clean_forecasts.isna().sum())

print("\nSaved cleaned copy to:")
print(clean_file)

print("\nRecords per run after cleaning:")
display(
    clean_forecasts.groupby("run_time_utc").size()
    .rename("hourly_records")
)


In [ ]:

# Download historical rainfall reference for July 2025

archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-07-28",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "GMT",
}

archive_response = requests.get(
    archive_url,
    params=archive_params,
    timeout=60,
)

print("Archive API status:", archive_response.status_code)

if archive_response.status_code == 200:
    archive_json = archive_response.json()

    july_reference_df = pd.DataFrame({
        "date": pd.to_datetime(
            archive_json["daily"]["time"]
        ).date,
        "reference_precipitation_mm": (
            archive_json["daily"]["precipitation_sum"]
        ),
        "reference_rainfall_mm": (
            archive_json["daily"]["rain_sum"]
        ),
    })

    reference_file = raw_dir / "ludhiana_july_2025_rainfall_reference.csv"
    july_reference_df.to_csv(reference_file, index=False)

    print("Rows downloaded:", len(july_reference_df))
    print("Missing values:")
    display(july_reference_df.isna().sum())

    print("\nFirst five rows:")
    display(july_reference_df.head())

    print("\nSaved to:")
    print(reference_file)

else:
    print(archive_response.text[:1500])


In [ ]:

# Match each forecast run's daily values to historical rainfall

clean_forecasts["valid_date"] = (
    clean_forecasts["valid_time_utc"].dt.date
)

daily_by_run = (
    clean_forecasts.groupby(
        ["run_time_utc", "valid_date"]
    )
    .agg(
        hours_available=("valid_time_utc", "count"),
        forecast_temperature_mean_c=("temperature_2m", "mean"),
        forecast_humidity_mean_pct=("relative_humidity_2m", "mean"),
        forecast_dew_point_mean_c=("dew_point_2m", "mean"),
        forecast_precipitation_sum_mm=("precipitation", "sum"),
        forecast_cloud_cover_mean_pct=("cloud_cover", "mean"),
        forecast_wind_speed_mean_kmh=("wind_speed_10m", "mean"),
    )
    .reset_index()
)

daily_by_run["valid_date"] = pd.to_datetime(
    daily_by_run["valid_date"]
).dt.date

july_reference_df["date"] = pd.to_datetime(
    july_reference_df["date"]
).dt.date

matched_forecasts = daily_by_run.merge(
    july_reference_df,
    left_on="valid_date",
    right_on="date",
    how="left",
    validate="many_to_one",
)

matched_forecasts = matched_forecasts.drop(columns=["date"])

print("Daily run-date combinations:", len(matched_forecasts))
print("Missing rainfall references:")
display(matched_forecasts["reference_rainfall_mm"].isna().sum())

print("\nSample matched records:")
display(matched_forecasts.head(10))

print("\nRows per forecast run:")
display(
    matched_forecasts.groupby("run_time_utc").size()
    .rename("daily_records")
)


In [ ]:

# Diagnose missing values in the merged daily dataset

print("Dataset shape:", matched_forecasts.shape)

print("\nMissing values by column:")
display(matched_forecasts.isna().sum())

print("\nRows with any missing values:")
display(
    matched_forecasts[
        matched_forecasts.isna().any(axis=1)
    ]
)

print("\nHours available per run and date:")
display(
    matched_forecasts[
        ["run_time_utc", "valid_date", "hours_available"]
    ].sort_values(["run_time_utc", "valid_date"])
)


In [ ]:
# Step: Inspect forecast rainfall against historical reference

display(
    matched_forecasts[
        [
            "run_time_utc",
            "valid_date",
            "hours_available",
            "forecast_precipitation_sum_mm",
            "reference_precipitation_mm",
            "reference_rainfall_mm",
        ]
    ].round(2)
)

print("Forecast rainfall summary:")
display(
    matched_forecasts["forecast_precipitation_sum_mm"].describe().round(2)
)

print("Historical reference rainfall summary:")
display(
    matched_forecasts["reference_rainfall_mm"].describe().round(2)
)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Keep rows where both rainfall values exist
comparison = matched_forecasts.dropna(
    subset=[
        "forecast_precipitation_sum_mm",
        "reference_rainfall_mm"
    ]
).copy()

# Actual reference and direct forecast
y_reference = comparison["reference_rainfall_mm"]
y_forecast = comparison["forecast_precipitation_sum_mm"]

mae = mean_absolute_error(y_reference, y_forecast)
rmse = np.sqrt(mean_squared_error(y_reference, y_forecast))

print("Number of valid daily comparisons:", len(comparison))
print(f"Direct forecast MAE:  {mae:.3f} mm")
print(f"Direct forecast RMSE: {rmse:.3f} mm")

print("\nInterpretation:")
print(f"On average, the absolute error is {mae:.2f} mm per day.")
print(f"RMSE is {rmse:.2f} mm and penalizes larger errors more heavily.")

In [ ]:
# Step: Check forecast dates and lead times

matched_forecasts["run_time_utc"] = pd.to_datetime(
    matched_forecasts["run_time_utc"]
)

matched_forecasts["valid_date"] = pd.to_datetime(
    matched_forecasts["valid_date"]
)

matched_forecasts["lead_days"] = (
    matched_forecasts["valid_date"].dt.normalize()
    - matched_forecasts["run_time_utc"].dt.normalize()
).dt.days

print("Forecast lead-time counts:")
display(matched_forecasts["lead_days"].value_counts().sort_index())

print("\nNumber of forecasts available per valid date:")
display(
    matched_forecasts.groupby("valid_date").size()
    .value_counts()
    .sort_index()
    .rename_axis("forecasts_per_date")
    .to_frame("number_of_dates")
)

print("\nSample with forecast lead time:")
display(
    matched_forecasts[
        [
            "run_time_utc",
            "valid_date",
            "lead_days",
            "forecast_precipitation_sum_mm",
            "reference_rainfall_mm",
        ]
    ].sort_values(["valid_date", "run_time_utc"]).head(15)
)

In [ ]:
# Verify the rainfall reference data and forecast values

print("Missing values in the reference column:")
print(matched_forecasts["reference_rainfall_mm"].isna().sum())

print("\nRows with missing reference rainfall:")
display(
    matched_forecasts.loc[
        matched_forecasts["reference_rainfall_mm"].isna(),
        ["run_time_utc", "valid_date", "reference_rainfall_mm"]
    ]
)

print("\nAll four run dates, with rainfall references:")
display(
    matched_forecasts[
        [
            "run_time_utc",
            "valid_date",
            "lead_days",
            "forecast_precipitation_sum_mm",
            "reference_rainfall_mm",
        ]
    ].sort_values(["run_time_utc", "valid_date"]).to_string(index=False)
)

In [ ]:
# Evaluate rainfall forecasts separately by lead time

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

lead_time_results = []

for lead, group in matched_forecasts.groupby("lead_days"):
    actual = group["reference_rainfall_mm"]
    forecast = group["forecast_precipitation_sum_mm"]

    lead_time_results.append({
        "lead_days": int(lead),
        "number_of_days": len(group),
        "MAE_mm": mean_absolute_error(actual, forecast),
        "RMSE_mm": np.sqrt(mean_squared_error(actual, forecast))
    })

lead_time_results = pd.DataFrame(lead_time_results)

display(lead_time_results.round(3))

print("\nTomorrow's forecast performance (lead time = 1 day):")
display(
    lead_time_results.loc[
        lead_time_results["lead_days"] == 1
    ].round(3)
)

In [ ]:
# Verify lead-time counts and evaluation results

print("Records per lead time:")
display(
    matched_forecasts.groupby("lead_days").size()
    .rename("actual_record_count")
    .reset_index()
)

print("\nEvaluation results:")
print(lead_time_results.to_string(index=False))

print("\nRows for tomorrow's lead time:")
display(
    matched_forecasts.loc[
        matched_forecasts["lead_days"] == 1,
        [
            "run_time_utc",
            "valid_date",
            "forecast_precipitation_sum_mm",
            "reference_rainfall_mm",
        ],
    ].sort_values("run_time_utc")
)

In [ ]:
from pathlib import Path
import pandas as pd

# Find previously saved historical forecast runs
folder = Path("../data/raw")

print("Notebook working directory:", Path.cwd())
print("\nCSV files in the raw-data folder:")

for file in sorted(folder.glob("*.csv")):
    print(f"{file.name} — {pd.read_csv(file, nrows=0).shape[1]} columns")

In [ ]:
from pathlib import Path
import pandas as pd

raw_folder = Path("../data/raw")

# Inspect the existing historical daily forecast
file_path = raw_folder / "ludhiana_historical_daily_forecast.csv"
historical_daily = pd.read_csv(file_path)

print("File:", file_path.name)
print("Shape:", historical_daily.shape)
print("\nColumns:")
print(historical_daily.columns.tolist())

display(historical_daily.head(5))
display(historical_daily.tail(5))

print("\nMissing values:")
display(historical_daily.isna().sum())

print("\nDate range:")
if "date" in historical_daily.columns:
    dates = pd.to_datetime(historical_daily["date"], errors="coerce")
    print("Start:", dates.min())
    print("End:", dates.max())
else:
    print("No column named 'date'. We will identify the correct date column.")

In [ ]:
from pathlib import Path
import pandas as pd

raw_folder = Path("../data/raw")

file_path = raw_folder / "ludhiana_rainfall_reference_2022_2025.csv"
rainfall_reference = pd.read_csv(file_path)

print("Dataset shape:", rainfall_reference.shape)
print("\nColumns:")
print(rainfall_reference.columns.tolist())

print("\nFirst five rows:")
display(rainfall_reference.head())

print("\nLast five rows:")
display(rainfall_reference.tail())

print("\nMissing values:")
display(rainfall_reference.isna().sum())

print("\nDate coverage:")
rainfall_reference["date"] = pd.to_datetime(rainfall_reference["date"])
print("Start:", rainfall_reference["date"].min())
print("End:", rainfall_reference["date"].max())

print("\nDuplicate dates:", rainfall_reference["date"].duplicated().sum())

In [ ]:

from pathlib import Path
import pandas as pd

raw_folder = Path("../data/raw")

file_path = raw_folder / "ludhiana_previous_day1_forecasts_hourly.csv"
hourly = pd.read_csv(file_path)

print("Dataset shape:", hourly.shape)
print("\nColumns:")
print(hourly.columns.tolist())

print("\nFirst five rows:")
display(hourly.head())

print("\nLast five rows:")
display(hourly.tail())

print("\nMissing values:")
display(hourly.isna().sum())

print("\nDate coverage:")
time_column = "time" if "time" in hourly.columns else "date"
hourly[time_column] = pd.to_datetime(hourly[time_column], errors="coerce")
print("Start:", hourly[time_column].min())
print("End:", hourly[time_column].max())

print("\nDuplicate timestamps:", hourly[time_column].duplicated().sum())


In [ ]:

# Check how much usable forecast data exists in each year

hourly["year"] = hourly["time"].dt.year

forecast_columns = [
    "temperature_2m_previous_day1",
    "relative_humidity_2m_previous_day1",
    "dew_point_2m_previous_day1",
    "precipitation_previous_day1",
    "cloud_cover_previous_day1",
    "wind_speed_10m_previous_day1",
]

coverage_rows = []

for year, group in hourly.groupby("year"):
    complete_rows = group[forecast_columns].notna().all(axis=1).sum()

    coverage_rows.append({
        "year": year,
        "hourly_rows": len(group),
        "complete_forecast_rows": complete_rows,
        "complete_percentage": round(
            complete_rows / len(group) * 100, 2
        ),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)

print("\nRows with all six forecast fields available:")
print(hourly[forecast_columns].notna().all(axis=1).sum())


In [ ]:

# Build clean daily forecast features for 2025

import pandas as pd
from pathlib import Path

forecast_columns = [
    "temperature_2m_previous_day1",
    "relative_humidity_2m_previous_day1",
    "dew_point_2m_previous_day1",
    "precipitation_previous_day1",
    "cloud_cover_previous_day1",
    "wind_speed_10m_previous_day1",
]

hourly_2025 = hourly.loc[
    hourly["time"].dt.year == 2025
].copy()

hourly_2025["date"] = hourly_2025["time"].dt.date

daily_forecasts = (
    hourly_2025.groupby("date")
    .agg(
        hours_available=("time", "count"),
        **{
            f"{column}_mean": (column, "mean")
            for column in forecast_columns
            if column != "precipitation_previous_day1"
        },
        forecast_precipitation_sum_mm=(
            "precipitation_previous_day1", "sum"
        ),
    )
    .reset_index()
)

# Keep only full days with all 24 hourly observations
daily_forecasts = daily_forecasts.loc[
    daily_forecasts["hours_available"] == 24
].copy()

# Remove days with missing forecast features
daily_forecasts = daily_forecasts.dropna()

output_path = Path("../data/processed/ludhiana_2025_daily_forecast_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
daily_forecasts.to_csv(output_path, index=False)

print("Daily forecast dataset shape:", daily_forecasts.shape)
print("Date range:", daily_forecasts["date"].min(), "to", daily_forecasts["date"].max())
print("Missing values:", daily_forecasts.isna().sum().sum())
print("Saved to:", output_path)

display(daily_forecasts.head())


In [ ]:

from pathlib import Path
import pandas as pd

processed_folder = Path("../data/processed")
raw_folder = Path("../data/raw")

# Load the daily forecast features and rainfall reference
daily_forecasts = pd.read_csv(
    processed_folder / "ludhiana_2025_daily_forecast_features.csv"
)

rainfall_reference = pd.read_csv(
    raw_folder / "ludhiana_rainfall_reference_2022_2025.csv"
)

# Normalize dates before merging
daily_forecasts["date"] = pd.to_datetime(daily_forecasts["date"]).dt.date
rainfall_reference["date"] = pd.to_datetime(
    rainfall_reference["date"]
).dt.date

# Select the 2025 rainfall reference
reference_2025 = rainfall_reference[
    pd.to_datetime(rainfall_reference["date"]).dt.year == 2025
].copy()

# Match forecast and reference dates
matched_2025 = daily_forecasts.merge(
    reference_2025[["date", "observed_rainfall_mm"]],
    on="date",
    how="inner",
    validate="one_to_one",
)

print("Matched dataset shape:", matched_2025.shape)
print("Date range:", matched_2025["date"].min(), "to", matched_2025["date"].max())
print("Missing values:", matched_2025.isna().sum().sum())
print("Duplicate dates:", matched_2025["date"].duplicated().sum())

display(matched_2025.head())


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = matched_2025["observed_rainfall_mm"]
forecast = matched_2025["forecast_precipitation_sum_mm"]

mae = mean_absolute_error(actual, forecast)
rmse = np.sqrt(mean_squared_error(actual, forecast))

print("FORECAST BASELINE EVALUATION")
print("-" * 40)
print(f"Dates evaluated: {len(matched_2025)}")
print(f"Reference rainy days: {(actual > 0.1).sum()}")
print(f"Forecast rainy days: {(forecast > 0.1).sum()}")
print(f"Reference total rainfall: {actual.sum():.2f} mm")
print(f"Forecast total precipitation: {forecast.sum():.2f} mm")
print(f"MAE: {mae:.3f} mm")
print(f"RMSE: {rmse:.3f} mm")


In [ ]:

from pathlib import Path

output_path = Path(
    "../data/processed/ludhiana_2025_matched_forecast_rainfall.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
matched_2025.to_csv(output_path, index=False)

print("Matched dataset saved successfully!")
print("File:", output_path.resolve())
print("Rows:", len(matched_2025))
print("Columns:", len(matched_2025.columns))


In [ ]:

import pandas as pd
from pathlib import Path

file_path = Path(
    "../data/processed/ludhiana_2025_matched_forecast_rainfall.csv"
)

df_check = pd.read_csv(file_path)

print("Dataset shape:", df_check.shape)
print("\nColumn names:")
for column in df_check.columns:
    print("-", column)

print("\nData types:")
print(df_check.dtypes)

print("\nRainfall target summary:")
print(df_check["observed_rainfall_mm"].describe())

print("\nFirst and last dates:")
print(df_check["date"].iloc[0], "to", df_check["date"].iloc[-1])


In [ ]:

# Check rainfall distribution and forecast quality

actual = df_check["observed_rainfall_mm"]
forecast = df_check["forecast_precipitation_sum_mm"]

print("RAINFALL DISTRIBUTION")
print("-" * 40)
print("Dry reference days (<= 0.1 mm):", (actual <= 0.1).sum())
print("Rainy reference days (> 0.1 mm):", (actual > 0.1).sum())
print("Maximum reference rainfall:", round(actual.max(), 2), "mm")
print("Maximum forecast precipitation:", round(forecast.max(), 2), "mm")

print("\nFEATURE CHECK")
print("-" * 40)
print("Forecast values below zero:", (forecast < 0).sum())
print("Missing values:", df_check.isna().sum().sum())
print("Duplicate dates:", df_check["date"].duplicated().sum())

print("\nRAINY-DAY DETECTION")
print("-" * 40)
actual_rainy = actual > 0.1
forecast_rainy = forecast > 0.1

print("Correct rainy-day detections:", (actual_rainy & forecast_rainy).sum())
print("Missed rainy days:", (actual_rainy & ~forecast_rainy).sum())
print("False rainy-day predictions:", (~actual_rainy & forecast_rainy).sum())


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

actual_rainy = (
    df_check["observed_rainfall_mm"] > 0.1
).astype(int)

forecast_rainy = (
    df_check["forecast_precipitation_sum_mm"] > 0.1
).astype(int)

print("DIRECT FORECAST — RAIN DETECTION METRICS")
print("-" * 45)
print(f"Accuracy:  {accuracy_score(actual_rainy, forecast_rainy):.3f}")
print(f"Precision: {precision_score(actual_rainy, forecast_rainy):.3f}")
print(f"Recall:    {recall_score(actual_rainy, forecast_rainy):.3f}")
print(f"F1-score:  {f1_score(actual_rainy, forecast_rainy):.3f}")

print("\nConfusion matrix:")
print("Rows = actual (dry, rainy); columns = predicted (dry, rainy)")
print(confusion_matrix(actual_rainy, forecast_rainy))

print("\nDetailed report:")
print(classification_report(
    actual_rainy,
    forecast_rainy,
    target_names=["Dry", "Rainy"],
    zero_division=0,
))


In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
)
import numpy as np

actual_amount = df_check["observed_rainfall_mm"]
forecast_amount = df_check["forecast_precipitation_sum_mm"]

actual_rain = (actual_amount > 0.1).astype(int)
predicted_rain = (forecast_amount > 0.1).astype(int)

metrics = pd.DataFrame([{
    "evaluation": "2025 historical direct-forecast baseline",
    "days_evaluated": len(df_check),
    "rain_detection_accuracy": accuracy_score(actual_rain, predicted_rain),
    "rain_detection_precision": precision_score(actual_rain, predicted_rain),
    "rain_detection_recall": recall_score(actual_rain, predicted_rain),
    "rain_detection_f1": f1_score(actual_rain, predicted_rain),
    "rainfall_amount_mae_mm": mean_absolute_error(actual_amount, forecast_amount),
    "rainfall_amount_rmse_mm": np.sqrt(
        mean_squared_error(actual_amount, forecast_amount)
    ),
}])

reports_folder = Path("../reports")
reports_folder.mkdir(parents=True, exist_ok=True)

report_path = reports_folder / "ludhiana_2025_direct_forecast_baseline.csv"
metrics.to_csv(report_path, index=False)

print("Baseline report saved:", report_path.resolve())
display(metrics.round(3))


In [ ]:

from pathlib import Path
import pandas as pd

file_path = Path(
    "../data/processed/ludhiana_2025_matched_forecast_rainfall.csv"
)

matched = pd.read_csv(file_path)

print("Matched dataset:", matched.shape)
print("\nAvailable columns:")
print(matched.columns.tolist())

print("\nSaved baseline report exists:",
      Path("../reports/ludhiana_2025_direct_forecast_baseline.csv").exists())

# Check whether the historical single-run experiment files exist
raw_folder = Path("../data/raw")

for filename in [
    "ludhiana_july_2025_historical_forecast_runs_clean.csv",
    "ludhiana_july_2025_rainfall_reference.csv",
]:
    path = raw_folder / filename
    print(f"{filename}: {path.exists()}")


In [ ]:

from pathlib import Path
import pandas as pd

raw_folder = Path("../data/raw")

runs_path = raw_folder / "ludhiana_july_2025_historical_forecast_runs_clean.csv"
reference_path = raw_folder / "ludhiana_july_2025_rainfall_reference.csv"

runs = pd.read_csv(runs_path)
reference = pd.read_csv(reference_path)

print("FORECAST RUN DATA")
print("-" * 40)
print("Shape:", runs.shape)
print("Columns:", runs.columns.tolist())
print("Missing values:\n", runs.isna().sum())
print("\nRun dates:")
print(sorted(runs["run_time_utc"].unique()))

print("\nRAINFALL REFERENCE")
print("-" * 40)
print("Shape:", reference.shape)
print("Columns:", reference.columns.tolist())
print("Date range:", reference["date"].min(), "to", reference["date"].max())

print("\nFirst forecast records:")
display(runs.head())


In [ ]:

# Parse forecast timestamps
runs["valid_time_utc"] = pd.to_datetime(runs["valid_time_utc"], utc=True)
runs["run_time_utc"] = pd.to_datetime(runs["run_time_utc"], utc=True)
reference["date"] = pd.to_datetime(reference["date"]).dt.date

# Compare each run date with its first and last valid timestamps
run_check = (
    runs.groupby("run_time_utc")
    .agg(
        first_valid_time=("valid_time_utc", "min"),
        last_valid_time=("valid_time_utc", "max"),
        hourly_records=("valid_time_utc", "count"),
    )
    .reset_index()
)

run_check["run_date_utc"] = run_check["run_time_utc"].dt.date

print("FORECAST RUN TIMING CHECK")
print("-" * 45)
display(run_check)

print("\nREFERENCE DATES")
print("First:", reference["date"].min())
print("Last:", reference["date"].max())
print("Reference rows:", len(reference))


In [ ]:

# Create the UTC calendar date for each forecast hour
runs["valid_date"] = runs["valid_time_utc"].dt.date

# Aggregate hourly forecast precipitation by run and valid date
daily_run_forecasts = (
    runs.groupby(["run_time_utc", "valid_date"])
    .agg(
        hours_available=("precipitation", "count"),
        forecast_precipitation_mm=("precipitation", "sum"),
        temperature_mean_c=("temperature_2m", "mean"),
        humidity_mean_pct=("relative_humidity_2m", "mean"),
        dew_point_mean_c=("dew_point_2m", "mean"),
        cloud_cover_mean_pct=("cloud_cover", "mean"),
        wind_speed_mean_kmh=("wind_speed_10m", "mean"),
    )
    .reset_index()
)

# Match each forecast date with its historical rainfall reference
daily_run_forecasts = daily_run_forecasts.merge(
    reference[["date", "reference_precipitation_mm", "reference_rainfall_mm"]],
    left_on="valid_date",
    right_on="date",
    how="left",
    validate="many_to_one",
).drop(columns=["date"])

print("Daily forecast rows:", len(daily_run_forecasts))
print("Missing reference values:",
      daily_run_forecasts["reference_rainfall_mm"].isna().sum())
print("\nHours available per forecast day:")
print(daily_run_forecasts["hours_available"].value_counts().sort_index())

print("\nFirst daily comparisons:")
display(daily_run_forecasts.head(10))


In [ ]:

# Find daily forecast records that do not contain all 24 hours
incomplete_days = daily_run_forecasts[
    daily_run_forecasts["hours_available"] != 24
].copy()

print("Incomplete forecast days:", len(incomplete_days))

display(
    incomplete_days[
        [
            "run_time_utc",
            "valid_date",
            "hours_available",
            "forecast_precipitation_mm",
            "reference_rainfall_mm",
        ]
    ]
)

print("\nComplete daily records:", (daily_run_forecasts["hours_available"] == 24).sum())
print("Total daily records:", len(daily_run_forecasts))


In [ ]:

from pathlib import Path

# Keep only complete 24-hour forecast days
complete_daily_forecasts = daily_run_forecasts[
    daily_run_forecasts["hours_available"] == 24
].copy()

# Save a separate clean file without overwriting the original
output_path = Path(
    "../data/processed/ludhiana_july_2025_complete_daily_forecasts.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

complete_daily_forecasts.to_csv(output_path, index=False)

print("Clean daily dataset saved!")
print("File:", output_path.resolve())
print("Rows:", len(complete_daily_forecasts))
print("Missing values:", complete_daily_forecasts.isna().sum().sum())
print(
    "All days have 24 hours:",
    (complete_daily_forecasts["hours_available"] == 24).all()
)

display(complete_daily_forecasts.head())


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = complete_daily_forecasts["reference_rainfall_mm"]
predicted = complete_daily_forecasts["forecast_precipitation_mm"]

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))

print("JULY COMPLETE-DAY FORECAST BASELINE")
print("-" * 45)
print("Days evaluated:", len(complete_daily_forecasts))
print(f"Reference rainfall total: {actual.sum():.2f} mm")
print(f"Forecast precipitation total: {predicted.sum():.2f} mm")
print(f"MAE: {mae:.3f} mm")
print(f"RMSE: {rmse:.3f} mm")


In [ ]:

print("JULY DATASET TARGET CHECK")
print("-" * 40)

actual = complete_daily_forecasts["reference_rainfall_mm"]

print("Days:", len(actual))
print("Dry days (<= 0.1 mm):", (actual <= 0.1).sum())
print("Rainy days (> 0.1 mm):", (actual > 0.1).sum())
print("Mean rainfall:", round(actual.mean(), 2), "mm")
print("Median rainfall:", round(actual.median(), 2), "mm")
print("Maximum rainfall:", round(actual.max(), 2), "mm")

print("\nForecast feature columns:")
print([
    "temperature_mean_c",
    "humidity_mean_pct",
    "dew_point_mean_c",
    "cloud_cover_mean_pct",
    "wind_speed_mean_kmh",
])


In [ ]:

print("DATA QUALITY CHECK")
print("-" * 40)

print("Dataset shape:", complete_daily_forecasts.shape)

print("\nMissing values:")
print(complete_daily_forecasts.isna().sum())

print("\nDuplicate run-date pairs:")
print(
    complete_daily_forecasts.duplicated(
        subset=["run_time_utc", "valid_date"]
    ).sum()
)

print("\nForecast runs:")
print(
    complete_daily_forecasts["run_time_utc"]
    .value_counts()
    .sort_index()
)

print("\nSample records:")
print(
    complete_daily_forecasts[
        [
            "run_time_utc",
            "valid_date",
            "forecast_precipitation_mm",
            "reference_rainfall_mm",
        ]
    ].head(10).to_string(index=False)
)


In [ ]:

import requests

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-07-07",
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "precipitation,"
        "cloud_cover,"
        "wind_speed_10m"
    ),
    "models": "ecmwf_ifs",
    "timezone": "UTC"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    result = response.json()
    print("Response keys:", list(result.keys()))
    print("Hourly variables:", list(result.get("hourly", {}).keys()))
    print("Number of hourly timestamps:", len(result.get("hourly", {}).get("time", [])))
    print("First timestamp:", result.get("hourly", {}).get("time", [None])[0])
else:
    print(response.text[:1000])


In [ ]:

import pandas as pd

hourly_data = result["hourly"]

historical_week = pd.DataFrame({
    "time": pd.to_datetime(hourly_data["time"], utc=True),
    "temperature_c": hourly_data["temperature_2m"],
    "humidity_pct": hourly_data["relative_humidity_2m"],
    "dew_point_c": hourly_data["dew_point_2m"],
    "precipitation_mm": hourly_data["precipitation"],
    "cloud_cover_pct": hourly_data["cloud_cover"],
    "wind_speed_kmh": hourly_data["wind_speed_10m"],
})

print("DATA CHECK")
print("-" * 40)
print("Shape:", historical_week.shape)
print("\nMissing values:")
print(historical_week.isna().sum())
print("\nDuplicate timestamps:", historical_week["time"].duplicated().sum())
print("\nTime range:", historical_week["time"].min(), "to", historical_week["time"].max())
print("\nHourly rainfall summary:")
print(historical_week["precipitation_mm"].describe().round(2))

print("\nDaily rainfall totals (mm):")
print(
    historical_week.assign(
        date=historical_week["time"].dt.strftime("%Y-%m-%d")
    ).groupby("date")["precipitation_mm"].sum().round(2)
)


In [ ]:

import requests
import pandas as pd

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-07-31",
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "precipitation,"
        "cloud_cover,"
        "wind_speed_10m"
    ),
    "models": "ecmwf_ifs",
    "timezone": "UTC"
}

response_month = requests.get(url, params=params, timeout=60)

print("HTTP status:", response_month.status_code)

if response_month.status_code == 200:
    month_json = response_month.json()
    month_hourly = month_json["hourly"]

    print("Hourly records:", len(month_hourly["time"]))
    print("Expected records:", 31 * 24)
    print("First timestamp:", month_hourly["time"][0])
    print("Last timestamp:", month_hourly["time"][-1])

    month_df = pd.DataFrame(month_hourly)
    print("\nMissing values by column:")
    print(month_df.isna().sum())
else:
    print(response_month.text[:1000])


In [ ]:

from pathlib import Path
import pandas as pd

# Create the raw-data folder if it does not exist
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_folder = project_root / "data" / "raw"
raw_folder.mkdir(parents=True, exist_ok=True)

# Convert API response into a DataFrame
july_2025_hourly = pd.DataFrame(month_json["hourly"])

# Save the original hourly weather records
save_path = raw_folder / "ludhiana_july_2025_historical_forecast.csv"
july_2025_hourly.to_csv(save_path, index=False)

print("Dataset saved successfully!")
print("File:", save_path.resolve())
print("Rows:", len(july_2025_hourly))
print("Columns:", list(july_2025_hourly.columns))
print("File exists:", save_path.exists())


In [ ]:

import requests
import pandas as pd
from pathlib import Path
import time

# Find the project folder
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_folder = project_root / "data" / "raw"
raw_folder.mkdir(parents=True, exist_ok=True)

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

months = [
    ("2025-07-01", "2025-07-31"),
    ("2025-08-01", "2025-08-31"),
    ("2025-09-01", "2025-09-30"),
    ("2025-10-01", "2025-10-31"),
    ("2025-11-01", "2025-11-30"),
    ("2025-12-01", "2025-12-31"),
]

hourly_fields = (
    "temperature_2m,"
    "relative_humidity_2m,"
    "dew_point_2m,"
    "precipitation,"
    "cloud_cover,"
    "wind_speed_10m"
)

saved_files = []

for start_date, end_date in months:
    filename = (
        f"ludhiana_historical_forecast_"
        f"{start_date[:7]}.csv"
    )
    save_path = raw_folder / filename

    # Reuse existing files instead of downloading again
    if save_path.exists():
        print("Already saved:", filename)
        saved_files.append(save_path)
        continue

    params = {
        "latitude": 30.9010,
        "longitude": 75.8573,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_fields,
        "models": "ecmwf_ifs",
        "timezone": "UTC",
    }

    response = requests.get(url, params=params, timeout=90)
    print(start_date[:7], "HTTP:", response.status_code)

    if response.status_code != 200:
        print("Request failed:", response.text[:300])
        continue

    month_json = response.json()
    month_df = pd.DataFrame(month_json["hourly"])

    expected_days = (
        pd.Timestamp(end_date) - pd.Timestamp(start_date)
    ).days + 1
    expected_rows = expected_days * 24

    if len(month_df) != expected_rows:
        print("Unexpected row count; not saving:", len(month_df))
        continue

    month_df.to_csv(save_path, index=False)
    saved_files.append(save_path)

    print("Saved:", filename, "| Rows:", len(month_df))
    time.sleep(1)

print("\nCOLLECTION SUMMARY")
print("Files available:", len(saved_files))
print("Folder:", raw_folder.resolve())


In [ ]:

from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_folder = project_root / "data" / "raw"
processed_folder = project_root / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

# Load the six monthly files
monthly_files = sorted(
    raw_folder.glob("ludhiana_historical_forecast_2025-*.csv")
)

print("Monthly files found:", len(monthly_files))

# Combine them
combined = pd.concat(
    [pd.read_csv(file) for file in monthly_files],
    ignore_index=True
)

# Standardize timestamps and sort chronologically
combined["time"] = pd.to_datetime(
    combined["time"], utc=True
)
combined = combined.sort_values("time").reset_index(drop=True)

# Quality checks
print("\nCOMBINED DATASET CHECK")
print("-" * 40)
print("Shape:", combined.shape)
print("First timestamp:", combined["time"].min())
print("Last timestamp:", combined["time"].max())
print("Duplicate timestamps:", combined["time"].duplicated().sum())

print("\nMissing values:")
print(combined.isna().sum())

# Save the combined dataset
output_path = (
    processed_folder /
    "ludhiana_historical_forecast_jul_dec_2025.csv"
)
combined.to_csv(output_path, index=False)

print("\nSaved:", output_path.resolve())


In [ ]:

import requests
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

raw_folder = project_root / "data" / "raw"
raw_folder.mkdir(parents=True, exist_ok=True)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-12-31",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "UTC",
}

response_rain = requests.get(url, params=params, timeout=90)

print("HTTP status:", response_rain.status_code)

if response_rain.status_code == 200:
    rain_json = response_rain.json()
    daily_data = rain_json["daily"]

    rainfall_reference = pd.DataFrame({
        "date": pd.to_datetime(daily_data["time"]),
        "reference_precipitation_mm": daily_data["precipitation_sum"],
        "reference_rainfall_mm": daily_data["rain_sum"],
    })

    print("Daily records:", len(rainfall_reference))
    print("First date:", rainfall_reference["date"].min())
    print("Last date:", rainfall_reference["date"].max())
    print("\nMissing values:")
    print(rainfall_reference.isna().sum())

    save_path = raw_folder / "ludhiana_rainfall_reference_jul_dec_2025.csv"
    rainfall_reference.to_csv(save_path, index=False)

    print("\nSaved:", save_path.resolve())
    print("\nFirst five records:")
    print(rainfall_reference.head().to_string(index=False))
else:
    print(response_rain.text[:1000])


In [ ]:

import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_folder = project_root / "data" / "processed"
raw_folder = project_root / "data" / "raw"

# Load the combined hourly weather dataset
hourly = pd.read_csv(
    processed_folder /
    "ludhiana_historical_forecast_jul_dec_2025.csv"
)

hourly["time"] = pd.to_datetime(hourly["time"], utc=True)
hourly["date"] = hourly["time"].dt.strftime("%Y-%m-%d")

# Convert hourly weather into daily features
daily_weather = (
    hourly.groupby("date")
    .agg(
        hours_available=("time", "count"),
        temperature_mean_c=("temperature_2m", "mean"),
        humidity_mean_pct=("relative_humidity_2m", "mean"),
        dew_point_mean_c=("dew_point_2m", "mean"),
        forecast_precipitation_mm=("precipitation", "sum"),
        cloud_cover_mean_pct=("cloud_cover", "mean"),
        wind_speed_mean_kmh=("wind_speed_10m", "mean"),
    )
    .reset_index()
)

# Load the matching daily rainfall reference
reference = pd.read_csv(
    raw_folder /
    "ludhiana_rainfall_reference_jul_dec_2025.csv"
)

reference["date"] = pd.to_datetime(reference["date"]).dt.strftime("%Y-%m-%d")

# Match by date
daily_dataset = daily_weather.merge(
    reference,
    on="date",
    how="inner",
    validate="one_to_one",
)

print("DAILY DATASET CHECK")
print("-" * 40)
print("Hourly rows:", len(hourly))
print("Daily weather rows:", len(daily_weather))
print("Matched daily rows:", len(daily_dataset))
print("Days with 24 hours:", (daily_dataset["hours_available"] == 24).sum())
print("Duplicate dates:", daily_dataset["date"].duplicated().sum())

print("\nMissing values:")
print(daily_dataset.isna().sum())

print("\nFirst five matched records:")
print(daily_dataset.head().round(2).to_string(index=False))

# Save the matched dataset
output_path = processed_folder / "ludhiana_matched_daily_dataset_jul_dec_2025.csv"
daily_dataset.to_csv(output_path, index=False)

print("\nSaved:", output_path.resolve())


In [ ]:

import numpy as np

print("FORECAST VS REFERENCE AUDIT")
print("-" * 45)

forecast = daily_dataset["forecast_precipitation_mm"]
reference = daily_dataset["reference_rainfall_mm"]

difference = forecast - reference

print("Matched days:", len(daily_dataset))
print("Exact matches:", np.isclose(forecast, reference).sum())
print("Days with different values:", (~np.isclose(forecast, reference)).sum())
print("Mean absolute difference (mm):", round(difference.abs().mean(), 3))
print("Root mean squared difference (mm):", round(np.sqrt((difference ** 2).mean()), 3))

print("\nForecast total (mm):", round(forecast.sum(), 2))
print("Reference total (mm):", round(reference.sum(), 2))

print("\nLargest differences:")
audit = daily_dataset[["date", "forecast_precipitation_mm", "reference_rainfall_mm"]].copy()
audit["absolute_difference_mm"] = (
    audit["forecast_precipitation_mm"] - audit["reference_rainfall_mm"]
).abs()

print(
    audit.sort_values("absolute_difference_mm", ascending=False)
    .head(10)
    .round(2)
    .to_string(index=False)
)


In [ ]:

print("HISTORICAL FORECAST API METADATA")
print("-" * 45)

print("Latitude:", month_json.get("latitude"))
print("Longitude:", month_json.get("longitude"))
print("Elevation:", month_json.get("elevation"))
print("Timezone:", month_json.get("timezone"))
print("UTC offset seconds:", month_json.get("utc_offset_seconds"))

print("\nHourly units:")
print(month_json.get("hourly_units"))

print("\nRequest configuration:")
print("Endpoint:", url)
print("Requested model:", params.get("models"))
print("Requested dates:", params.get("start_date"), "to", params.get("end_date"))

print("\nReference API configuration:")
print("Reference endpoint:", "https://archive-api.open-meteo.com/v1/archive")
print("Reference fields:", "precipitation_sum, rain_sum")


In [ ]:

import requests

print("VERIFY API SOURCES")
print("-" * 40)

forecast_url = (
    "https://historical-forecast-api.open-meteo.com/v1/forecast"
)

forecast_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-07-03",
    "hourly": "precipitation",
    "models": "ecmwf_ifs",
    "timezone": "UTC",
}

forecast_response = requests.get(
    forecast_url, params=forecast_params, timeout=60
)

archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-01",
    "end_date": "2025-07-03",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "UTC",
}

archive_response = requests.get(
    archive_url, params=archive_params, timeout=60
)

print("Forecast endpoint:", forecast_response.url)
print("Forecast HTTP:", forecast_response.status_code)
print("Archive endpoint:", archive_response.url)
print("Archive HTTP:", archive_response.status_code)

if forecast_response.status_code == 200 and archive_response.status_code == 200:
    forecast_json = forecast_response.json()
    archive_json = archive_response.json()

    forecast_df = pd.DataFrame({
        "time": pd.to_datetime(
            forecast_json["hourly"]["time"], utc=True
        ),
        "forecast_precipitation_mm": forecast_json["hourly"]["precipitation"],
    })

    forecast_df["date"] = forecast_df["time"].dt.strftime("%Y-%m-%d")

    forecast_daily = forecast_df.groupby("date")[
        "forecast_precipitation_mm"
    ].sum()

    archive_daily = pd.Series(
        archive_json["daily"]["precipitation_sum"],
        index=archive_json["daily"]["time"],
        name="archive_precipitation_mm",
    )

    print("\nThree-day comparison:")
    print(pd.concat([forecast_daily, archive_daily], axis=1))
else:
    print("One or both requests failed.")
    print(forecast_response.text[:300])
    print(archive_response.text[:300])


In [ ]:

forecast_params.update({
    "start_date": "2025-07-15",
    "end_date": "2025-07-17",
})

archive_params.update({
    "start_date": "2025-07-15",
    "end_date": "2025-07-17",
})

forecast_response = requests.get(
    forecast_url, params=forecast_params, timeout=60
)
archive_response = requests.get(
    archive_url, params=archive_params, timeout=60
)

print("Forecast HTTP:", forecast_response.status_code)
print("Archive HTTP:", archive_response.status_code)

if forecast_response.status_code == 200 and archive_response.status_code == 200:
    forecast_json = forecast_response.json()
    archive_json = archive_response.json()

    hourly_df = pd.DataFrame({
        "time": pd.to_datetime(
            forecast_json["hourly"]["time"], utc=True
        ),
        "forecast_precipitation_mm": forecast_json["hourly"]["precipitation"],
    })
    hourly_df["date"] = hourly_df["time"].dt.strftime("%Y-%m-%d")

    forecast_daily = hourly_df.groupby("date")[
        "forecast_precipitation_mm"
    ].sum()

    archive_daily = pd.Series(
        archive_json["daily"]["precipitation_sum"],
        index=archive_json["daily"]["time"],
        name="archive_precipitation_mm",
    )

    comparison = pd.concat([forecast_daily, archive_daily], axis=1)
    comparison["absolute_difference_mm"] = (
        comparison["forecast_precipitation_mm"]
        - comparison["archive_precipitation_mm"]
    ).abs()

    print("\nComparison:")
    print(comparison.round(2))
else:
    print("Request failed. Check the HTTP status above.")


In [ ]:

import requests
import pandas as pd

run_url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

run_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-15",
    "end_date": "2025-07-17",
    "hourly": "precipitation",
    "models": "ecmwf_ifs",
    "timezone": "UTC",
}

run_response = requests.get(
    run_url, params=run_params, timeout=60
)

print("Historical forecast API status:", run_response.status_code)

if run_response.status_code == 200:
    run_json = run_response.json()
    print("Response keys:", list(run_json.keys()))
    print("Hourly records:", len(run_json["hourly"]["time"]))
    print("Requested model:", run_params["models"])
    print(
        "\nNote: this endpoint returns a historical forecast series, "
        "not necessarily one fixed initialization run."
    )
else:
    print(run_response.text[:500])


In [ ]:

import requests

single_run_url = "https://api.open-meteo.com/v1/ecmwf"

single_run_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "hourly": "temperature_2m,precipitation",
    "start_date": "2025-07-14",
    "end_date": "2025-07-16",
    "timezone": "UTC",
}

single_response = requests.get(
    single_run_url,
    params=single_run_params,
    timeout=60,
)

print("Single-run API HTTP status:", single_response.status_code)
print("Requested URL:", single_response.url)

if single_response.status_code == 200:
    single_json = single_response.json()
    print("Response keys:", list(single_json.keys()))
    print("Hourly fields:", list(single_json.get("hourly", {}).keys()))
    print(
        "Hourly records:",
        len(single_json.get("hourly", {}).get("time", [])),
    )
    print(
        "First timestamp:",
        single_json.get("hourly", {}).get("time", [None])[0],
    )
else:
    print(single_response.text[:1000])


In [ ]:

import requests

historical_run_url = "https://previous-runs-api.open-meteo.com/v1/forecast"

historical_run_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "hourly": "temperature_2m,precipitation",
    "models": "ecmwf_ifs",
    "forecast_days": 3,
    "past_days": 0,
    "timezone": "UTC",
}

historical_run_response = requests.get(
    historical_run_url,
    params=historical_run_params,
    timeout=60,
)

print("HTTP status:", historical_run_response.status_code)
print("Response:", historical_run_response.text[:1500])


In [ ]:

import requests

docs_url = "https://open-meteo.com/en/docs/historical-forecast-api"

docs_response = requests.get(docs_url, timeout=30)

print("Documentation HTTP status:", docs_response.status_code)
print("Documentation page accessible:", docs_response.status_code == 200)

if docs_response.status_code == 200:
    print("Page title/content preview:")
    print(docs_response.text[:500])
else:
    print(docs_response.text[:500])


In [ ]:

import requests

single_run_url = "https://single-runs-api.open-meteo.com/v1/forecast"

single_run_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "run": "2025-07-14T00:00",
    "hourly": "temperature_2m,precipitation",
    "models": "ecmwf_ifs",
    "forecast_days": 3,
    "timezone": "UTC",
}

single_run_response = requests.get(
    single_run_url,
    params=single_run_params,
    timeout=60,
)

print("HTTP status:", single_run_response.status_code)

if single_run_response.status_code == 200:
    single_run_data = single_run_response.json()
    print("Returned model run:", single_run_params["run"])
    print("Response keys:", list(single_run_data.keys()))
    print("Hourly records:", len(single_run_data.get("hourly", {}).get("time", [])))
    print("First timestamps:", single_run_data.get("hourly", {}).get("time", [])[:3])
    print("First precipitation values:", single_run_data.get("hourly", {}).get("precipitation", [])[:3])
else:
    print("API response:", single_run_response.text[:1500])


In [ ]:

import pandas as pd

hourly_run_df = pd.DataFrame(single_run_data["hourly"])

hourly_run_df["time"] = pd.to_datetime(hourly_run_df["time"], utc=True)

print("Dataset shape:", hourly_run_df.shape)
print("\nFirst five rows:")
print(hourly_run_df.head().to_string(index=False))

print("\nMissing values:")
print(hourly_run_df.isna().sum())

print("\nHourly records by date:")
print(hourly_run_df.groupby(hourly_run_df["time"].dt.date).size())

print("\nPrecipitation summary (mm):")
print(hourly_run_df["precipitation"].describe())


In [ ]:

# Keep only rows with a recorded precipitation value
valid_run_df = hourly_run_df.dropna(subset=["precipitation"]).copy()

# Create a date column in UTC
valid_run_df["date"] = valid_run_df["time"].dt.date

# Summarize each day
daily_run_df = (
    valid_run_df.groupby("date")
    .agg(
        hours_with_precipitation=("precipitation", "count"),
        mean_temperature_c=("temperature_2m", "mean"),
        forecast_precipitation_mm=("precipitation", "sum"),
    )
    .reset_index()
)

print("Daily forecast summary:")
print(daily_run_df.to_string(index=False))

print("\nTotal valid hourly records:", len(valid_run_df))
print("Days covered:", daily_run_df["date"].nunique())
print(
    "\nNote: daily precipitation totals may be incomplete "
    "because one hourly value was missing."
)


In [ ]:

import requests
import pandas as pd

archive_url = "https://archive-api.open-meteo.com/v1/archive"

archive_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "start_date": "2025-07-14",
    "end_date": "2025-07-16",
    "daily": "precipitation_sum,rain_sum",
    "timezone": "UTC",
}

archive_response = requests.get(
    archive_url,
    params=archive_params,
    timeout=60,
)

print("Archive API HTTP status:", archive_response.status_code)

if archive_response.status_code == 200:
    archive_data = archive_response.json()

    reference_df = pd.DataFrame(archive_data["daily"])
    reference_df["date"] = pd.to_datetime(reference_df["time"]).dt.date
    reference_df = reference_df.rename(
        columns={
            "precipitation_sum": "reference_precipitation_mm",
            "rain_sum": "reference_rainfall_mm",
        }
    ).drop(columns=["time"])

    print("\nHistorical rainfall reference:")
    print(reference_df.to_string(index=False))
else:
    print(archive_response.text[:1000])


In [ ]:

print("Forecast vs historical reference:")

columns_to_show = [
    "date",
    "hours_with_precipitation",
    "forecast_precipitation_mm",
    "reference_precipitation_mm",
    "absolute_error_mm",
]

print(comparison_df[columns_to_show].to_string(index=False))

print(
    "\nMean absolute error (mm):",
    round(comparison_df["absolute_error_mm"].mean(), 3),
)

print("Matched days:", len(comparison_df))

print(
    "Note: July 14 has one missing hourly precipitation value, "
    "so its forecast total may be incomplete."
)


In [ ]:

from pathlib import Path
import pandas as pd

runs_path = Path(
    "../data/raw/ludhiana_july_2025_historical_forecast_runs_clean.csv"
)

if not runs_path.exists():
    runs_path = Path(
        "data/raw/ludhiana_july_2025_historical_forecast_runs_clean.csv"
    )

if runs_path.exists():
    runs_df = pd.read_csv(runs_path)

    print("File:", runs_path)
    print("Shape:", runs_df.shape)
    print("\nColumns:", runs_df.columns.tolist())

    if "run_time_utc" in runs_df.columns:
        print("\nAvailable initialization times:")
        print(sorted(runs_df["run_time_utc"].dropna().unique()))

    print("\nMissing values:")
    print(runs_df.isna().sum().to_string())
else:
    print("File not found from the current notebook directory.")
    print("Current directory:", Path.cwd())


In [ ]:

import requests

test_run_params = {
    "latitude": 30.9010,
    "longitude": 75.8573,
    "run": "2025-07-29T00:00",
    "hourly": (
        "temperature_2m,relative_humidity_2m,"
        "dew_point_2m,precipitation,cloud_cover,wind_speed_10m"
    ),
    "models": "ecmwf_ifs",
    "forecast_days": 7,
    "timezone": "UTC",
}

test_run_response = requests.get(
    "https://single-runs-api.open-meteo.com/v1/forecast",
    params=test_run_params,
    timeout=60,
)

print("HTTP status:", test_run_response.status_code)

if test_run_response.status_code == 200:
    test_run_data = test_run_response.json()
    test_hourly = test_run_data.get("hourly", {})

    print("Hourly records:", len(test_hourly.get("time", [])))
    print("First timestamp:", test_hourly.get("time", [None])[0])
    print("Last timestamp:", test_hourly.get("time", [None])[-1])
    print(
        "Missing precipitation values:",
        sum(v is None for v in test_hourly.get("precipitation", [])),
    )
else:
    print(test_run_response.text[:1000])


In [ ]:

import requests
import pandas as pd
from pathlib import Path
import time

latitude = 30.9010
longitude = 75.8573

run_dates = pd.date_range(
    start="2025-07-01",
    end="2025-12-31",
    freq="7D",
)

hourly_variables = (
    "temperature_2m,relative_humidity_2m,"
    "dew_point_2m,precipitation,cloud_cover,wind_speed_10m"
)

all_runs = []
failed_runs = []

for run_date in run_dates:
    run_time = run_date.strftime("%Y-%m-%dT00:00")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "run": run_time,
        "hourly": hourly_variables,
        "models": "ecmwf_ifs",
        "forecast_days": 7,
        "timezone": "UTC",
    }

    try:
        response = requests.get(
            "https://single-runs-api.open-meteo.com/v1/forecast",
            params=params,
            timeout=60,
        )

        if response.status_code == 200:
            hourly = response.json().get("hourly", {})
            run_df = pd.DataFrame(hourly)

            if not run_df.empty and "time" in run_df.columns:
                run_df["run_time_utc"] = run_time
                all_runs.append(run_df)
                print(run_time, "records:", len(run_df))
            else:
                failed_runs.append((run_time, "No hourly records"))
        else:
            failed_runs.append((run_time, response.text[:200]))

    except Exception as exc:
        failed_runs.append((run_time, str(exc)))

    time.sleep(0.3)

if all_runs:
    collected_runs_df = pd.concat(all_runs, ignore_index=True)
    collected_runs_df["time"] = pd.to_datetime(
        collected_runs_df["time"], utc=True
    )

    output_path = Path(
        "../data/raw/ludhiana_ecmwf_weekly_runs_jul_dec_2025.csv"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)

    collected_runs_df.to_csv(output_path, index=False)

    print("\nSaved to:", output_path.resolve())
    print("Total records:", len(collected_runs_df))
    print("Successful runs:", len(all_runs))
    print("Failed runs:", len(failed_runs))

    if failed_runs:
        print("\nFailed runs:")
        for item in failed_runs:
            print(item)
else:
    print("No runs were collected.")
    print("Failures:", failed_runs)


In [ ]:

from pathlib import Path
import pandas as pd

runs_path = Path(
    "../data/raw/ludhiana_ecmwf_weekly_runs_jul_dec_2025.csv"
)

runs_df = pd.read_csv(runs_path)

print("Dataset shape:", runs_df.shape)
print("\nColumns:")
print(runs_df.columns.tolist())

print("\nMissing values:")
print(runs_df.isna().sum().to_string())

print("\nNumber of unique forecast runs:")
print(runs_df["run_time_utc"].nunique())

print("\nForecast records per run:")
print(runs_df.groupby("run_time_utc").size().to_string())

print("\nFirst five rows:")
print(runs_df.head().to_string(index=False))

print("\nDuplicate run-and-time records:")
print(
    runs_df.duplicated(
        subset=["run_time_utc", "time"]
    ).sum()
)


In [ ]:

# Find records where precipitation is missing
missing_precip_df = runs_df[
    runs_df["precipitation"].isna()
].copy()

print("Total dataset rows:", len(runs_df))
print("Unique forecast runs:", runs_df["run_time_utc"].nunique())
print("Missing precipitation records:", len(missing_precip_df))

print("\nMissing precipitation by forecast run:")
print(
    missing_precip_df.groupby("run_time_utc")
    .size()
    .to_string()
)

print("\nExact missing records:")
print(
    missing_precip_df[
        ["run_time_utc", "time", "temperature_2m"]
    ].to_string(index=False)
)


In [ ]:

# Keep a copy of the original dataset
runs_clean_df = runs_df.copy()

# Remove only rows where precipitation is missing
runs_clean_df = runs_clean_df.dropna(
    subset=["precipitation"]
).copy()

print("Original rows:", len(runs_df))
print("Rows after cleaning:", len(runs_clean_df))
print("Rows removed:", len(runs_df) - len(runs_clean_df))

print("\nRemaining missing values:")
print(runs_clean_df.isna().sum().to_string())

print("\nRecords per forecast run after cleaning:")
print(runs_clean_df.groupby("run_time_utc").size().to_string())

print(
    "\nNote: each run now has 167 valid precipitation hours. "
    "Daily totals for dates affected by a removed hour may be incomplete."
)


In [ ]:

# Parse timestamps
runs_clean_df["time"] = pd.to_datetime(
    runs_clean_df["time"], utc=True
)

runs_clean_df["run_time_utc"] = pd.to_datetime(
    runs_clean_df["run_time_utc"], utc=True
)

# Calendar date of each forecast's valid time
runs_clean_df["valid_date"] = runs_clean_df["time"].dt.date

# Aggregate each run and valid date separately
daily_forecasts_df = (
    runs_clean_df.groupby(["run_time_utc", "valid_date"])
    .agg(
        hours_available=("precipitation", "count"),
        temperature_mean_c=("temperature_2m", "mean"),
        humidity_mean_pct=("relative_humidity_2m", "mean"),
        dew_point_mean_c=("dew_point_2m", "mean"),
        forecast_precipitation_mm=("precipitation", "sum"),
        cloud_cover_mean_pct=("cloud_cover", "mean"),
        wind_speed_mean_kmh=("wind_speed_10m", "mean"),
    )
    .reset_index()
)

print("Daily forecast dataset shape:", daily_forecasts_df.shape)
print("\nFirst 10 daily records:")
print(daily_forecasts_df.head(10).to_string(index=False))

print("\nHours available per daily record:")
print(daily_forecasts_df["hours_available"].value_counts().sort_index())

print("\nUnique forecast runs:", daily_forecasts_df["run_time_utc"].nunique())
print("Unique valid dates:", daily_forecasts_df["valid_date"].nunique())


In [ ]:

from pathlib import Path
import pandas as pd

reference_path = Path(
    "../data/raw/ludhiana_rainfall_reference_jul_dec_2025.csv"
)

if not reference_path.exists():
    reference_path = Path(
        "data/raw/ludhiana_rainfall_reference_jul_dec_2025.csv"
    )

reference_jul_dec_df = pd.read_csv(reference_path)

reference_jul_dec_df["date"] = pd.to_datetime(
    reference_jul_dec_df["date"]
).dt.date

# Match each forecast valid date to the reference date
matched_daily_df = daily_forecasts_df.merge(
    reference_jul_dec_df,
    left_on="valid_date",
    right_on="date",
    how="left",
    validate="many_to_one",
)

print("Matched dataset shape:", matched_daily_df.shape)

print("\nMissing reference values:")
print(
    matched_daily_df[
        ["reference_precipitation_mm", "reference_rainfall_mm"]
    ].isna().sum().to_string()
)

print("\nRows with 24 hours available:")
print((matched_daily_df["hours_available"] == 24).sum())

print("\nFirst 10 matched records:")
print(
    matched_daily_df[
        [
            "run_time_utc",
            "valid_date",
            "hours_available",
            "forecast_precipitation_mm",
            "reference_rainfall_mm",
        ]
    ].head(10).to_string(index=False)
)


In [ ]:

missing_reference_df = matched_daily_df[
    matched_daily_df["reference_rainfall_mm"].isna()
]

print("Rows missing rainfall reference:", len(missing_reference_df))

print("\nAffected forecast dates and runs:")
print(
    missing_reference_df[
        ["run_time_utc", "valid_date", "hours_available"]
    ].to_string(index=False)
)

print("\nAvailable reference date range:")
print("Start:", min(reference_jul_dec_df["date"]))
print("End:", max(reference_jul_dec_df["date"]))

print("\nReference rows:", len(reference_jul_dec_df))
print("Duplicate reference dates:", reference_jul_dec_df["date"].duplicated().sum())


In [ ]:

evaluation_df = matched_daily_df[
    (matched_daily_df["hours_available"] == 24)
    & (matched_daily_df["reference_rainfall_mm"].notna())
].copy()

evaluation_df = evaluation_df.sort_values(
    ["run_time_utc", "valid_date"]
).reset_index(drop=True)

print("Clean evaluation dataset shape:", evaluation_df.shape)
print("\nMissing values:")
print(
    evaluation_df[
        ["forecast_precipitation_mm", "reference_rainfall_mm"]
    ].isna().sum().to_string()
)

print("\nHours available:")
print(evaluation_df["hours_available"].value_counts().to_string())

print("\nForecast runs:", evaluation_df["run_time_utc"].nunique())
print("Unique target dates:", evaluation_df["valid_date"].nunique())

print("\nFirst 10 clean records:")
print(
    evaluation_df[
        [
            "run_time_utc",
            "valid_date",
            "hours_available",
            "forecast_precipitation_mm",
            "reference_rainfall_mm",
        ]
    ].head(10).to_string(index=False)
)


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_true = evaluation_df["reference_rainfall_mm"]
y_pred = evaluation_df["forecast_precipitation_mm"]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
bias = (y_pred - y_true).mean()

print("BASELINE FORECAST EVALUATION")
print("-" * 35)
print(f"Records evaluated: {len(evaluation_df)}")
print(f"MAE:  {mae:.3f} mm")
print(f"RMSE: {rmse:.3f} mm")
print(f"Bias: {bias:.3f} mm")

print("\nInterpretation:")
print("MAE = average absolute rainfall error.")
print("RMSE = penalizes large rainfall errors more heavily.")
print("Bias > 0 means the forecast overestimates rainfall on average.")
print("Bias < 0 means the forecast underestimates rainfall on average.")


In [ ]:

evaluation_df["run_date"] = (
    evaluation_df["run_time_utc"].dt.date
)

evaluation_df["lead_days"] = (
    pd.to_datetime(evaluation_df["valid_date"])
    - pd.to_datetime(evaluation_df["run_date"])
).dt.days

print("Forecast lead-time distribution:")
print(evaluation_df["lead_days"].value_counts().sort_index().to_string())

print("\nRecords per target date:")
print(evaluation_df.groupby("valid_date").size().value_counts().sort_index().to_string())

print("\nNumber of unique target dates:", evaluation_df["valid_date"].nunique())
print("Number of forecast records:", len(evaluation_df))

print("\nSample with lead time:")
print(
    evaluation_df[
        ["run_time_utc", "valid_date", "lead_days",
         "forecast_precipitation_mm", "reference_rainfall_mm"]
    ].head(12).to_string(index=False)
)


In [ ]:

tomorrow_eval_df = evaluation_df[
    evaluation_df["lead_days"] == 1
].copy()

y_true_1day = tomorrow_eval_df["reference_rainfall_mm"]
y_pred_1day = tomorrow_eval_df["forecast_precipitation_mm"]

mae_1day = mean_absolute_error(y_true_1day, y_pred_1day)
rmse_1day = np.sqrt(mean_squared_error(y_true_1day, y_pred_1day))
bias_1day = (y_pred_1day - y_true_1day).mean()

print("TOMORROW-ONLY FORECAST EVALUATION")
print("-" * 40)
print("Records:", len(tomorrow_eval_df))
print("Unique target dates:", tomorrow_eval_df["valid_date"].nunique())
print(f"MAE:  {mae_1day:.3f} mm")
print(f"RMSE: {rmse_1day:.3f} mm")
print(f"Bias: {bias_1day:.3f} mm")

print("\nFirst 10 tomorrow-only records:")
print(
    tomorrow_eval_df[
        ["run_time_utc", "valid_date",
         "forecast_precipitation_mm", "reference_rainfall_mm"]
    ].head(10).to_string(index=False)
)


In [ ]:

rain = tomorrow_eval_df["reference_rainfall_mm"]

print("REFERENCE RAINFALL DISTRIBUTION")
print("-" * 40)
print("Total records:", len(rain))
print("Dry days (less than 0.1 mm):", (rain < 0.1).sum())
print("Rainy days (0.1 mm or more):", (rain >= 0.1).sum())
print("Moderate/heavy days (5 mm or more):", (rain >= 5).sum())
print("Heavy rainfall days (20 mm or more):", (rain >= 20).sum())

print("\nRainfall statistics (mm):")
print(rain.describe().round(2).to_string())


In [ ]:

print("Available columns:")
for column in daily_forecasts_df.columns:
    print("-", column)

print("\nDataset shape:", daily_forecasts_df.shape)

print("\nMissing values by column:")
print(daily_forecasts_df.isna().sum().to_string())

print("\nSample daily forecast features:")
print(daily_forecasts_df.head(5).to_string(index=False))


In [ ]:

from pathlib import Path

output_path = Path("../data/processed/ludhiana_forecast_evaluation_jul_dec_2025.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

evaluation_df.to_csv(output_path, index=False)

print("Evaluation dataset saved successfully!")
print("File:", output_path.resolve())
print("Rows:", len(evaluation_df))
print("Columns:", len(evaluation_df.columns))
print("File exists:", output_path.exists())


In [ ]:

from pathlib import Path
import pandas as pd

history_path = Path("../data/raw/ludhiana_rainfall_reference_2022_2025.csv")

if not history_path.exists():
    history_path = Path("data/raw/ludhiana_rainfall_reference_2022_2025.csv")

history_df = pd.read_csv(history_path)

print("Historical dataset shape:", history_df.shape)
print("\nColumns:")
print(history_df.columns.tolist())

print("\nFirst 5 rows:")
print(history_df.head().to_string(index=False))

print("\nDate range:")
print("Start:", history_df["date"].min())
print("End:", history_df["date"].max())

print("\nMissing values:")
print(history_df.isna().sum().to_string())


In [ ]:

history_df["date"] = pd.to_datetime(history_df["date"])

print("Historical rainfall quality check")
print("-" * 40)

print("Duplicate dates:", history_df["date"].duplicated().sum())
print(
    "Negative observed rainfall values:",
    (history_df["observed_rainfall_mm"] < 0).sum()
)
print(
    "Negative observed precipitation values:",
    (history_df["observed_precipitation_mm"] < 0).sum()
)

print("\nRainfall summary (mm):")
print(history_df["observed_rainfall_mm"].describe().round(2).to_string())

print("\nRainfall categories:")
rain = history_df["observed_rainfall_mm"]
print("Dry days (< 0.1 mm):", (rain < 0.1).sum())
print("Rainy days (>= 0.1 mm):", (rain >= 0.1).sum())
print("Days >= 5 mm:", (rain >= 5).sum())
print("Days >= 20 mm:", (rain >= 20).sum())

print(
    "\nPrecipitation and rainfall columns identical:",
    history_df["observed_precipitation_mm"].equals(
        history_df["observed_rainfall_mm"]
    )
)


In [ ]:

historical_model_df = history_df.sort_values("date").copy()

historical_model_df["tomorrow_rainfall_mm"] = (
    historical_model_df["observed_rainfall_mm"].shift(-1)
)

print("Prepared historical dataset:", historical_model_df.shape)

print("\nFirst 7 rows:")
print(
    historical_model_df[
        ["date", "observed_rainfall_mm", "tomorrow_rainfall_mm"]
    ].head(7).to_string(index=False)
)

print("\nLast 3 rows:")
print(
    historical_model_df[
        ["date", "observed_rainfall_mm", "tomorrow_rainfall_mm"]
    ].tail(3).to_string(index=False)
)

print(
    "\nMissing tomorrow targets:",
    historical_model_df["tomorrow_rainfall_mm"].isna().sum()
)


In [ ]:

historical_model_df["month"] = historical_model_df["date"].dt.month
historical_model_df["day_of_year"] = historical_model_df["date"].dt.dayofyear
historical_model_df["day_of_week"] = historical_model_df["date"].dt.dayofweek

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

historical_model_df["season"] = (
    historical_model_df["month"].apply(get_season)
)

print("Calendar features added.")
print("\nFirst 10 rows:")
print(
    historical_model_df[
        ["date", "month", "day_of_year", "day_of_week", "season",
         "observed_rainfall_mm", "tomorrow_rainfall_mm"]
    ].head(10).to_string(index=False)
)

print("\nSeason counts:")
print(historical_model_df["season"].value_counts().to_string())


In [ ]:

training_data_df = historical_model_df.dropna(
    subset=["tomorrow_rainfall_mm"]
).copy()

training_data_df = training_data_df.sort_values("date").reset_index(drop=True)

print("Training dataset prepared!")
print("Rows:", len(training_data_df))
print("Columns:", len(training_data_df.columns))

print("\nMissing values:")
print(training_data_df.isna().sum().to_string())

print("\nDate range:")
print("Start:", training_data_df["date"].min().date())
print("End:", training_data_df["date"].max().date())

print("\nTarget summary:")
print(training_data_df["tomorrow_rainfall_mm"].describe().round(2).to_string())


In [ ]:

feature_columns = [
    "observed_rainfall_mm",
    "month",
    "day_of_year",
    "day_of_week",
    "season",
]

target_column = "tomorrow_rainfall_mm"

X = training_data_df[feature_columns].copy()
y = training_data_df[target_column].copy()

print("Features selected successfully!")
print("\nFeature columns:", feature_columns)
print("Target column:", target_column)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature sample:")
print(X.head(5).to_string(index=False))

print("\nTarget sample:")
print(y.head(5).to_string(index=False))


In [ ]:

# RainfallAI — Check whether tomorrow's target uses the next calendar day

import pandas as pd

check_df = historical_model_df.sort_values("date").copy()
check_df["next_date"] = check_df["date"].shift(-1)

check_df["days_until_next_row"] = (
    check_df["next_date"] - check_df["date"]
).dt.days

print("First date:", check_df["date"].min())
print("Last date:", check_df["date"].max())

print(
    "\nRows where the next observation is not exactly one day later:",
    int(
        (
            check_df["days_until_next_row"].notna()
            & (check_df["days_until_next_row"] != 1)
        ).sum()
    )
)

print("\nExamples of date gaps:")
display(
    check_df.loc[
        check_df["days_until_next_row"].notna()
        & (check_df["days_until_next_row"] != 1),
        ["date", "next_date", "days_until_next_row"]
    ].head(10)
)

print("\nMissing tomorrow targets:", check_df["tomorrow_rainfall_mm"].isna().sum())
